### Libraries used

In [1]:
# Libraries used for Analysis Spatial Points Pattern

# Manipulation of list and math calculs...
import numpy as np
import math
# Use for statistics tests
from scipy.stats import chi2, norm
# Use for the NN-D and KDE part
from scipy.spatial.distance import pdist, squareform
from scipy.spatial import distance_matrix, ConvexHull
from scipy.stats import gaussian_kde
# Print library
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from shapely.geometry import Point, Polygon
# Library for the application of the Analysis (Quadrat, K-functions...)
import pointpats
from pointpats import QStatistic, distance_statistics, PointPattern
# Library to import file, '.til' here
import tifffile as tiff
# Library to analyse and recover data on image
from skimage import measure
from sklearn.cluster import DBSCAN
# Library to extract or generate data in CSV file
import pandas as pd
import os
import re

### Functions of calcul (here kNN)

In [2]:
# Function to calculate and store (in list) all nn-d values and the corresponding point indices and the corresponding pairs indices
def unique_nearest_neighbor_distances(points):
    """
    Parameters:
        points: A 2D list of generated points (from the model)

    Returns:
        nn_distances : list of every unique nn-d values
        indices_pts : list of corresponding points indices of the nn-d
        recorded_pairs : list of tuple of point indices of the nn-d (d(i,j))
    """
    
    # Calculate all pairwise distances between each point
    distances = pdist(points)
    # Convert 'distances' into a square matrix
    dist_matrix = squareform(distances)
    
    # Initialize lists to store nn-d and corresponding point indices
    nn_distances = []
    indices_pts = []
    
    # Browse each point to find the nn-d value without duplicates
    num_points = len(points)
    recorded_pairs = set()
    
    for i in range(num_points):
        # Obtain distances from point i to other points
        distances_from_i = dist_matrix[i]
        # Ignore the distance of point i from itself (which is always 0)
        distances_from_i[i] = np.inf
        # Find the associated index of the nearest neighbor of point i
        j = np.argmin(distances_from_i)
        nearest_distance = distances_from_i[j]
        
        # Ensure that we only record the distance if (i, j) or (j, i) has not been recorded yet
        if (i, j) not in recorded_pairs and (j, i) not in recorded_pairs:
            nn_distances.append(nearest_distance)
            indices_pts.append(i)
            recorded_pairs.add((i, j))
    
    # Return the list of nn-d and the list of the corresponding point indices and pairs indices
    return nn_distances, indices_pts, recorded_pairs

### Functions to realize statistic tests

- Skellam statistic test :

$$S_m = 2 \lambda \pi \sum_{i = 1}^{M} D_{i}^2$$

It's chi-square ($\chi^2$) distributed with $2m$ degrees of freedom. . 
Hence this statistic provides a test of the CSR Hypothesis based on nearest neighbors.

- Clark-Evans test :

While Skellam’s statistic can be used to construct tests, it follows from the Central Limit Theorem (CLT) that independent sums of identically distributed random variables are approximately normally distributed. Hence the most common test of the CSR Hypothesis based on nearest neighbors involves a normal approximation to the sample mean of D , as defined by :

$$D_m = \frac{1}{m} \sum_{i = 1}^{M} D_i$$

$D_m$ must be approximately normally distributed under the CSR Hypothesis with mean and variance as follows :

$$D_m ~ N[\frac{1}{2\sqrt \lambda},\frac{4 - \pi}{m(4\lambda \pi)}]$$

then one can test the CSR Hypothesis by constructing the following standardized sample mean:

$$Z_m = \frac{D_m - \mu}{\sigma}$$

In [3]:
# Function which is doing the Clark-Evans test and/or the Skellam one
def Clark_Evans_Skellam_function(lambda_,points,nn_distances,method):
    """
    Parameters:
        lambda_ : density of the model
        points: A 2D list of generated points (from the model)
        nn_distances : list of every unique nn-d values
        method : value to know which test to do

    Returns:
        S_m : Skellam's statistic
        critical_value : threshold value of Skellam' statistic
        alpha : significance level
        z_m : Chi-2 test value
        d_m : Mean of nn-d
    """
    
    # Skellam statistic test :
    if method == 0 or method == 2:
        # Occurrence of NN-Distance values
        nb_nn_d = len(nn_distances)
        
        # Calculate S_m (Skellam Statistic) :
        S_m = 0

        for i in range(nb_nn_d):
            S_m += nn_distances[i]**2
        S_m = 2 * lambda_ * np.pi * S_m

        # Test of the value under CSR Hypothesis
        alpha = 0.05  # Significance level
        df = 2 * nb_nn_d  # degree of freedom
        critical_value = chi2.ppf(1 - alpha, df)  # critical value of chi-2 (chi-square) distribution
        
        if method == 0:
            # Return Skellam value, the critical one and the level alpha
            return S_m, critical_value, alpha
    
    # Clark-Evans test :
    if method == 1 or method == 2:
        # Recover the nn-d of the points
        nn_distances, _, _ = unique_nearest_neighbor_distances(points)
        # Implement the number of nn-d 
        nb_nn_d = len(nn_distances) 
        # CLT Parameters for the normal distribution (mu : Expected Value, sigma : standard deviation/ sigma**2 : variance)
        mu = 1 / (2 * np.sqrt(lambda_))
        sigma = np.sqrt((4 - np.pi) / (len(points) * 4 * lambda_ * np.pi))

        # Part of Clark-Evans test
        
        # Calculate D_m
        d_m = 1 / nb_nn_d * np.sum(nn_distances)
        # D_m = 9.037

        # Calculate z_m
        z_m = (d_m - mu) / sigma

        # Return the z_m value to correspond to a normal distribution (N(0,1)), d_m value the mean of all nn-d
        return z_m, d_m
    
    # Return the outputs of both test
    return S_m, critical_value, alpha, z_m, d_m

# Calculating function of Z_m & D_m mean with a random number of points selected
def Clark_Evans_simulation(num_points,points,lambda_,N):
    """
    Parameters:
        num_points : Number of points of the model
        points: A 2D list of generated points (from the model)
        lambda_ : density of the model
        N : Number of simulation

    Returns:
        all_Z_m : List of z_m value
        all_D_m : List of d_m value
    """
    
    # Initialize the lists of z_m value and d_m value
    all_Z_m = []
    all_D_m = []
    
    # Simulate N times the algorithm to have N value of z_m and d_m
    for _ in range(N):
        # Random selection of 'half' of the points
        selected_indices = np.random.choice(num_points,2 * num_points // 3, replace=False)
        selected_points = points[selected_indices]
        
        # Calculation of nn-d
        nn_distances, _, _ = unique_nearest_neighbor_distances(selected_points)
        
        # Initialization of lists Z_m and D_m for this iteration
        Z_m = 0
        D_m = 0
        
        # Executing the Clark_Evans_Skellam_function
        Z_m, D_m = Clark_Evans_Skellam_function(lambda_, selected_points, nn_distances, method=1)
        
        # Storage of the new results done above
        all_Z_m.append(Z_m)
        all_D_m.append(D_m)
    
    # Return the lists of Z_m values and D_m values
    return all_Z_m, all_D_m
        
# Function of the behaviour of z_m, with the p-value method / two-tailed and 'conclusion on model type'
def p_value_behaviour(z_m,alpha,z_alpha_2):
    """
    Parameters:
        z_m : normal distribution value of our pattern
        alpha : significance level (in %)
        z_alpha_2 : significance level under normal distribution
    """
    
    # Reject CSR Hypothesis if |z_m| < z_alpha/2
    if np.abs(z_m) < z_alpha_2:
        print("\nDo not reject the CSR Hypothesis")
    else:
        print("\nReject the CSR Hypothesis")
        # Determination of the model in relation to z_m value
        if z_m < 0:
            print("Clustering model")
        else:
            print("Dispersion model")
        
    # Calculation of the p-value according to the value z_m
    # Clustering or random case
    if z_m < 0:
        p_value_cluster = norm.cdf(z_m)
        p_value_dispersion = np.inf
    # Dispersed or random case
    else:
        p_value_cluster = np.inf
        p_value_dispersion = 1 - norm.cdf(z_m)
    p_value_sides = 2 * norm.cdf(- np.abs(z_m))

    # Display the results of each p-value
    print(f"\np-value Cluster: {p_value_cluster}")
    print(f"p-value Dispersion: {p_value_dispersion}")
    print(f"p-value Two-Tails: {p_value_sides}")
    
    # Storage of the previous values
    p_value = [p_value_cluster,p_value_dispersion,p_value_sides]
    status = 0
    
    # Conclusion on the model according to the p-value
    # Dispersed or clustering model conclusion
    for i in range(0,len(p_value)):
        if p_value[i] < alpha:
            print("\nReject the CSR Hypothesis")
            if i == 0:
                print("Clustering Model")
            elif i == 1:
                print("Dispersion Model")
            else:
                print("Not a random model")
            status = 1

    # Random model conclusion
    if status == 0:
        print("\nDo not reject the CSR Hypothesis")
        print("Possible random model")

### Display function of the model with specific features

In [4]:
# Display function of Spatial points pattern and the associated NN-Distances
def display_spatial_points_analysis(coordonnees, area_dim, nb_points, ind_nn_d, nn_distances, tuple_dist):
    """
    Parameters:
        coordonnees : A 2D list of generated points (from the model)
        area_dim : Dimension of the area (assumed to be square)
        nb_points : Number of points of the model
        ind_nn_d : list of corresponding points indices of the nn-d
        nn_distances : list of every unique nn-d values
        tuple_dist : list of every couple which represent the unique nn-d values
    """

    # Visualization of the pattern following the Poisson Approximation
    plt.figure(figsize=(8, 8))
    plt.scatter(coordonnees[:, 0], coordonnees[:, 1], c='blue', marker='o', s=10, label='Points')

    # Map out the circles to visualize the NN-Distances stored and calculated before
    for i in range(len(ind_nn_d)):  # You can add a third value 'k' to display 1 circle on k in the graph
        # Here, we print only the circle of the points registered in the list of indices of nn-d 'ind_nn_d'
        circle = plt.Circle(coordonnees[ind_nn_d[i]], nn_distances[i], color='red', fill=False, linestyle='dotted')
        # Add the display of the circle number i to the others done before
        plt.gca().add_patch(circle)
        
    # Plot the lines representing the NN-Distances
    for (i, j) in tuple_dist:
        pt1 = coordonnees[i]
        pt2 = coordonnees[j]
        plt.plot([pt1[0], pt2[0]], [pt1[1], pt2[1]], '--', color="lightgreen")

    # Legend for NN-D circles
    legend_circle = Patch(color='red', linestyle='dotted', fill=False, label='Circles of NN-D')
    # Legend for the pattern points
    legend_point = Line2D([0], [0], color='blue', marker='o', linestyle='', label='Pattern Points')
    # Legend for the NN-D lines
    legend_line = Line2D([0], [0], color='lightgreen', linestyle='--', label='Lines of NN-D')

    # Commands to print the Spatial Points Model
    plt.title('Spatial Poisson Process (under CSR or forced) and kNN (K-Nearest Neighbors)')
    plt.xlabel('Coordinate X')
    plt.ylabel('Coordinate Y')
    plt.legend(handles=[legend_point, legend_circle, legend_line])
    # plt.grid(True)
    plt.show()

### Quadrat method function

This method can help us to determine the kind of points pattern that we have.

To determine it we are using the Khi-2 test ($\chi^2$) :

$$\chi^2 = \frac{s^2}{n_{m}} (m - 1)$$

where $m - 1$ is the degree of freedom.

The model is clustered if $\frac{s^2}{n_m} \ge 1$.

The model is dispersed if $\frac{s^2}{n_m} \le 1$.

Else it's a random model

So, in the following code we calculate :

$$ \frac{\chi^2}{(m - 1)} = \frac{s^2}{n_m}$$

In [5]:
# Function to print the quadrat method
def quadrat_method(points):
    """
    Parameters:
        points : A 2D list of generated points (from the model)
    """
    
    # Quadrat analysis from our pattern
    qstat = QStatistic(points,nx=3,ny=3)
    qstat.plot()
    
    # Informations related to quadrat model
    # Chi-2 value
    print(f"\nChi-2 value quadrat (Chi-2): {qstat.chi2}")
    # Degree of freedom (number of boxes minus 1) (m - 1)
    print(f"Degrees of freedom (m-1): {qstat.df}")
    # Threshold to determine the type of model (Chi2 = (s**2 / n_mean) * (m - 1))
    print(f"s**2 / n_mean : {qstat.chi2/qstat.df}")
    # P-value of the quadrat method
    print(f"Chi-2 p-value : {qstat.chi2_pvalue}\n")
    
    # Conclusion related to the previous results
    # Dispersed case
    if qstat.chi2/qstat.df < 1:
        # Threshold to affirm that the model is dispersed
        if qstat.chi2/qstat.df < 0.8:
            print("The pattern is certainly dispersed")
        # Possibility to be random or dispersed
        else:
            print("The pattern is possibly dispersed or random")
    # Random case
    elif qstat.chi2/qstat.df == 1:
        print("The pattern is built randomly")
    # Clustered case
    else:
        # Threshold to affirm that the model is clustered
        if qstat.chi2/qstat.df < 6:
            print("The pattern is possibly clustered or random")
        # Possibility to random or clustered
        else:
            print("The pattern is certainly clustered")

### Ripley's functions (F,G,J,K,L...) implementations and display, the general version (without taking care of edge effects)

The Ripley's function are used to analyze and conclude on the possible type of pattern supplied.

We know and decide to study the model with the help of 5 Ripley's function :

- F functon $F(h)$: 

This function measures the nearest neighbor distance from a set of known randomly-distributed points to a point in the observed pattern. This reflects a between-pattern measure of dispersion, where one pattern is completely spatially random and the other pattern is our observed pattern.

For a randomly simulated point pattern of size $N_r$, $F(d)$ function is :

$$F(h) = \frac{1}{N_r} \sum_{k}^{N_r} I_h (d_{ik} < h)$$

- G function $G(h)$:

This function works like a distribution function as we can see sometimes in statistics. For our given distance value $h$, $G(h)$ is the propotion of kNN distances that are less than this $h$ value. We already have above a code which explain and compute the calculation of this kNN. So we have $G(h)$ function such as :

$$G(h) = \frac{1}{N} \sum_{i = 1}^{N} I_h (d_{ij} < h)$$

- K functon $K(h)$:

This function is close to calculate the expected points presence ($\lambda K(h)$) within the circle of radius $h$ center on point $i$.

$$K(h) = \frac{1}{n\lambda} \sum_{i=1}^{n} \sum_{j \neq i} I_h (d_{ij} < h)$$

- L function $L(h)$:

This function is simply the linearisation of the K-function :

$$L(h) = \sqrt \frac{K(h)}{\pi}$$

An H-function can be defined as follow but useless to add it to our Ripley's function analysis :

$$H(h) = L(h) - h$$

- J function $J(h)$:

This function is simply the ratio between two of the previous function that we just see before $G$ and $F$:

$$J(h) = \frac{1 - G(h)}{1 - F(h)}$$


These are the Ripley's functions implement below

A comment part on their implemention with the related library 'pointpats' is below with the function code that corrected the problem (it will be presented inside this part).

In [6]:
# Function to store the distance where the observation value is above the simulations
def get_significant_distances(support, statistic, simulations):
    """
    Parameters:
        support : 1D array of distances
        statistic : 1D array of observed statistics
        simulations : 2D array of simulated statistics (each row corresponds to a different simulation)

    Returns:
        significant_distances : list of distances where the observed statistic is strictly greater than all simulated statistics
    """
    significant_distances = []
    for i, dist in enumerate(support):
        if statistic[i] > np.max(simulations[:, i]):
            significant_distances.append(dist)
    return significant_distances

In [7]:
# Display function of Ripley's G function (if red line above the blue one -> clustering, under dispersion)
def display_Ripley_G(points, max_dist, data_info, hull_pattern=None, save_path='Data_Pasteur/16h29_5h18_exp191_rpr/Plotting_Analysis'):
    """
    Parameters:
        points : A 2D list of generated points (from the model)
        max_dist : maximum distance of the nn-d
        data_info : additional information to specify the data (e.g., 'data1', 'data2', etc.)
        hull_pattern: bounding box, scipy.spatial.ConvexHull, shapely.geometry.Polygon the hull used to construct a random sample pattern 
        save_path : path where the figure should be saved
    """
    
    # Ensure the save directory exists
    full_save_path = os.path.join(save_path, data_info)
    os.makedirs(full_save_path, exist_ok=True)
    
    # Calculation of the Ripley's G function
    g_test = distance_statistics.g_test(points, support=40, hull=hull_pattern, keep_simulations=True, n_simulations=1000)
    
    # Get significant distances where the observed G function is greater than all simulations
    significant_distances = get_significant_distances(g_test.support, g_test.statistic, g_test.simulations)
    
    # Print significant distances
    print("Significant distances where observed G function is greater than all simulations:")
    print(significant_distances)
    
    # Parameters for the display
    f, ax = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw=dict(width_ratios=(6, 3)))
    
    # Plot all the simulations with very fine lines (black lines)
    ax[0].plot(g_test.support, g_test.simulations.T, color="k", alpha=0.01)
    
    # and show the average of simulations (blue line)
    ax[0].plot(
        g_test.support,
        np.median(g_test.simulations, axis=0),
        color="cyan",
        label="median simulation",
    )

    # and the observed pattern's G function (red line)
    ax[0].plot(g_test.support, g_test.statistic, label="observed", color="red")

    # clean up labels and axes
    ax[0].set_xlabel("distance")
    ax[0].set_ylabel("% of nearest neighbor\ndistances shorter")
    ax[0].legend()
    # max_dist represent the maximum limit where it means to calculate the Ripley's function (max_dist = largest kNN)
    ax[0].set_xlim(0, max_dist)
    ax[0].set_title(r"Ripley's $G(d)$ function")

    # plot the pattern itself on the right frame
    ax[1].scatter(*points.T)

    # and clean up labels and axes there, too
    ax[1].set_xticks([])
    ax[1].set_yticks([])
    ax[1].set_xticklabels([])
    ax[1].set_yticklabels([])
    ax[1].set_title("Pattern")
    f.tight_layout()
    
    # Save the figure
    file_path = os.path.join(full_save_path, 'G_function.png')
    plt.savefig(file_path)
    plt.show()
    
    return significant_distances
    
# Display function of Ripley's F function (if red line under the blue one -> clustering, above dispersion)
def display_Ripley_F(points, max_dist, data_info, hull_pattern=None, save_path='Data_Pasteur/16h29_5h18_exp191_rpr/Plotting_Analysis'):
    """
    Parameters:
        points : A 2D list of generated points (from the model)
        max_dist : maximum distance of the nn-d
        data_info : additional information to specify the data (e.g., 'data1', 'data2', etc.)
        hull_pattern: bounding box, scipy.spatial.ConvexHull, shapely.geometry.Polygon the hull used to construct a random sample pattern 
        save_path : path where the figure should be saved
    """
    
    # Ensure the save directory exists
    full_save_path = os.path.join(save_path, data_info)
    os.makedirs(full_save_path, exist_ok=True)
    
    # Calculation of the Ripley's F function
    f_test = distance_statistics.f_test(points, support=40, hull=hull_pattern, keep_simulations=True, n_simulations=1000)
    
    # Get significant distances where the observed G function is greater than all simulations
    significant_distances = get_significant_distances(f_test.support, f_test.statistic, f_test.simulations)
    
    # Print significant distances
    print("Significant distances where observed F function is greater than all simulations:")
    print(significant_distances)
    
    # Parameters for the display
    f, ax = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw=dict(width_ratios=(6, 3)))

    # plot all the simulations with very fine lines (black lines)
    ax[0].plot(f_test.support, f_test.simulations.T, color="k", alpha=0.01)
    
    # and show the average of simulations (blue line)
    ax[0].plot(
        f_test.support,
        np.median(f_test.simulations, axis=0),
        color="cyan",
        label="median simulation",
    )


    # and the observed pattern's F function (red line)
    ax[0].plot(f_test.support, f_test.statistic, label="observed", color="red")

    # clean up labels and axes
    ax[0].set_xlabel("distance")
    ax[0].set_ylabel("% of nearest point in pattern\ndistances shorter")
    ax[0].legend()
    # max_dist represent the maximum limit where it means to calculate the Ripley's function (max_dist = largest kNN)
    ax[0].set_xlim(0, max_dist)
    ax[0].set_title(r"Ripley's $F(d)$ function")

    # plot the pattern itself on the right frame
    ax[1].scatter(*points.T)

    # and clean up labels and axes there, too
    ax[1].set_xticks([])
    ax[1].set_yticks([])
    ax[1].set_xticklabels([])
    ax[1].set_yticklabels([])
    ax[1].set_title("Pattern")
    f.tight_layout()
    
    # Save the figure
    file_path = os.path.join(full_save_path, 'F_function.png')
    plt.savefig(file_path)
    plt.show()
    
# Display function of Ripley's K function (if red line above the blue one -> clustering, under dispersion)
def display_Ripley_K(points, max_dist, data_info, hull_pattern=None, save_path='Data_Pasteur/16h29_5h18_exp191_rpr/Plotting_Analysis'):
    """
    Parameters:
        points : A 2D list of generated points (from the model)
        max_dist : maximum distance of the nn-d
        data_info : additional information to specify the data (e.g., 'data1', 'data2', etc.)
        hull_pattern: bounding box, scipy.spatial.ConvexHull, shapely.geometry.Polygon the hull used to construct a random sample pattern 
        save_path : path where the figure should be saved
    """
    
    # Ensure the save directory exists
    full_save_path = os.path.join(save_path, data_info)
    os.makedirs(full_save_path, exist_ok=True)
    
    # Calculation of the Ripley's K function
    k_test = distance_statistics.k_test(points, support=40, hull=hull_pattern, keep_simulations=True, n_simulations=1000)
    
    # Get significant distances where the observed G function is greater than all simulations
    significant_distances = get_significant_distances(k_test.support, k_test.statistic, k_test.simulations)
    
    # Print significant distances
    print("Significant distances where observed K function is greater than all simulations:")
    print(significant_distances)

    # Parameters for the display
    fig, ax = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw=dict(width_ratios=(6, 3)))

    # plot all the simulations with very fine lines (black lines)
    ax[0].plot(k_test.support, k_test.simulations.T, color="k", alpha=0.01)
    
    # and show the average of simulations (blue line)
    ax[0].plot(
        k_test.support,
        np.median(k_test.simulations, axis=0),
        color="cyan",
        label="median simulation",
    )

    # and the observed pattern's K function (red line)
    ax[0].plot(k_test.support, k_test.statistic, label="observed", color="red")

    # clean up labels and axes
    ax[0].set_xlabel("Distance")
    ax[0].set_ylabel("K(d) value")
    ax[0].legend()
    # max_dist represent the maximum limit where it means to calculate the Ripley's function (max_dist = largest kNN)
    ax[0].set_xlim(0, max_dist)
    ax[0].set_title(r"Ripley's $K(d)$ function")

    # plot the pattern itself on the right frame
    ax[1].scatter(*points.T)

    # and clean up labels and axes there, too
    ax[1].set_xticks([])
    ax[1].set_yticks([])
    ax[1].set_xticklabels([])
    ax[1].set_yticklabels([])
    ax[1].set_title("Pattern")
    
    fig.tight_layout()
    
    # Save the figure
    file_path = os.path.join(full_save_path, 'K_function.png')
    plt.savefig(file_path)
    plt.show()
    
# Display function of Ripley's L function (if red line above the blue one -> clustering, under dispersion)
def display_Ripley_L(points, max_dist, data_info, hull_pattern=None, save_path='Data_Pasteur/16h29_5h18_exp191_rpr/Plotting_Analysis'):
    """
    Parameters:
        points : A 2D list of generated points (from the model)
        max_dist : maximum distance of the nn-d
        data_info : additional information to specify the data (e.g., 'data1', 'data2', etc.)
        hull_pattern: bounding box, scipy.spatial.ConvexHull, shapely.geometry.Polygon the hull used to construct a random sample pattern 
        save_path : path where the figure should be saved
    """
    
    # Ensure the save directory exists
    full_save_path = os.path.join(save_path, data_info)
    os.makedirs(full_save_path, exist_ok=True)
    
    # Calculation of the Ripley's L function
    l_test = distance_statistics.l_test(points, support=40, hull=hull_pattern, keep_simulations=True, n_simulations=1000)
    
    # Get significant distances where the observed G function is greater than all simulations
    significant_distances = get_significant_distances(l_test.support, l_test.statistic, l_test.simulations)
    
    # Print significant distances
    print("Significant distances where observed L function is greater than all simulations:")
    print(significant_distances)

    # Parameters for the display
    fig, ax = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw=dict(width_ratios=(6, 3)))

    # plot all the simulations with very fine lines (black lines)
    ax[0].plot(l_test.support, l_test.simulations.T, color="k", alpha=0.01)
    
    # and show the average of simulations (blue line)
    ax[0].plot(
        l_test.support,
        np.median(l_test.simulations, axis=0),
        color="cyan",
        label="median simulation",
    )

    # and the observed pattern's K function (red line)
    ax[0].plot(l_test.support, l_test.statistic, label="observed", color="red")

    # clean up labels and axes
    ax[0].set_xlabel("Distance")
    ax[0].set_ylabel("L(d) value")
    ax[0].legend()
    # max_dist represent the maximum limit where it means to calculate the Ripley's function (max_dist = largest kNN)
    ax[0].set_xlim(0, max_dist)
    ax[0].set_title(r"Ripley's $L(d)$ function")

    # plot the pattern itself on the right frame
    ax[1].scatter(*points.T)

    # and clean up labels and axes there, too
    ax[1].set_xticks([])
    ax[1].set_yticks([])
    ax[1].set_xticklabels([])
    ax[1].set_yticklabels([])
    ax[1].set_title("Pattern")
    
    fig.tight_layout()
    
    # Save the figure
    file_path = os.path.join(full_save_path, 'L_function.png')
    plt.savefig(file_path)
    plt.show()
    
# Display function of Ripley's J function (if the line above the threshold equal to 1 -> dispersion, otherwise clustering)
# J function works as a ratio (J(d) = 1, random pattern)
def display_Ripley_J(points, max_dist, data_info, hull_pattern=None, save_path='Data_Pasteur/16h29_5h18_exp191_rpr/Plotting_Analysis'):
    """
    Parameters:
        points : A 2D list of generated points (from the model)
        max_dist : maximum distance of the nn-d
        data_info : additional information to specify the data (e.g., 'data1', 'data2', etc.)
        hull_pattern: bounding box, scipy.spatial.ConvexHull, shapely.geometry.Polygon the hull used to construct a random sample pattern 
        save_path : path where the figure should be saved
    """
    
    # Ensure the save directory exists
    full_save_path = os.path.join(save_path, data_info)
    os.makedirs(full_save_path, exist_ok=True)
    
    # Calculation of the Ripley's J function
    j_test = distance_statistics.j_test(points, support=40, hull=hull_pattern)
    
    # Get significant distances where the observed G function is greater than all simulations
    significant_distances = get_significant_distances(j_test.support, j_test.statistic, j_test.simulations)
    
    # Print significant distances
    print("Significant distances where observed J function is greater than all simulations:")
    print(significant_distances)

    # Parameters for the display
    fig, ax = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw=dict(width_ratios=(6, 3)))

    # and the observed pattern's J function (red line)
    ax[0].plot(j_test.support, j_test.statistic, label="observed", color="orange")
    
    ax[0].axhline(1, linestyle=':',color='k')

    # clean up labels and axes
    ax[0].set_xlabel("Distance")
    ax[0].set_ylabel("J(d) value")
    ax[0].legend()
    # max_dist represent the maximum limit where it means to calculate the Ripley's function (max_dist = largest kNN)
    ax[0].set_xlim(0, max_dist)
    ax[0].set_title(r"Ripley's $J(d)$ function")

    # plot the pattern itself on the right frame
    ax[1].scatter(*points.T)

    # and clean up labels and axes there, too
    ax[1].set_xticks([])
    ax[1].set_yticks([])
    ax[1].set_xticklabels([])
    ax[1].set_yticklabels([])
    ax[1].set_title("Pattern")
    
    fig.tight_layout()

    # Save the figure
    file_path = os.path.join(full_save_path, 'J_function.png')
    plt.savefig(file_path)
    plt.show()
            
# Function to display all Ripley's functions (F,G,J,K,L)
def Ripley_dispaly_all(points,max_dist,data_info,hull=None):
    """
    Parameters:
        points : A 2D list of generated points (from the model)
        max_dist : MAX distance of the nn-d list for the printed interval of Ripley's functions
        data_info : additional information to specify the data (e.g., 'data1', 'data2', etc.)
        hull: bounding box, scipy.spatial.ConvexHull, shapely.geometry.Polygon the hull used to construct a random sample pattern  
    """
    
    # K-function (F,G,K,L,J) analysis and display
    print("\nDisplay of Ripley's functions :")
    _ = display_Ripley_G(points,max_dist,data_info,hull_pattern=hull)
    display_Ripley_F(points,max_dist,data_info,hull_pattern=hull)
    display_Ripley_K(points,max_dist,data_info,hull_pattern=hull)
    display_Ripley_L(points,max_dist,data_info,hull_pattern=hull)
    display_Ripley_J(points,max_dist,data_info,hull_pattern=hull)

### Ripley's correction function (and Besag's correction), analytic correction implementations

To apply the Ripley's correcction on the associated functions, we need to calculate a value which will take more in count the possible edge effects that the points of the model can face.

First, we have to calculate this weight ($w_{ij}$) for each pair of points. We will have a matrix which store the weight of the point $i$ associated to the point $j$.

$w_{ij}$ is the ratio of the circle which the center is located at the point $(x_i,y_i)$ with a radius of distance $d(i,j) = d_{ij} = d$. It's the ratio of the circle circumference inside the area 'region' R ($p_{ij} (C_{i} | R)$) on the total circle circumference ($p^{tot}_{ij} = 2\pi d_{ij}$):

$$ w_{ij} = \frac{p_{ij} (C_{i} | R)}{p^{tot}_{ij}} $$ 

for Ripley's Correction,
where $$0 \le w_{ij} \le 1$$,

The Besag's correction is based on the same principe of a same ratio, but this time instead of being the ratio of the circumference inside the region R on the total circumference. We have to calculate the ratio of the area inside the region R ($A_{ij}$) on the total area of the circle ($A^{tot}_{ij} = \pi d_{ij}^2$):

$$ w_{ij} = \frac{A_{ij} (C_{i} | R)}{A^{tot}_{ij}} $$ 

for Besag correction,
where $$0 \le w_{ij} \le 1$$, and with this equation between this two mathematical correction :

$$ 0 \le w_{RC} \le w_{BC} \le 1 $$

with $w_{RC}$ and $w_{BC}$ respectively the weight under the Ripley's Correction and the weight under Besag Correction.

To calculate this with a regular region (square or rectangle), here are the formula of the proportion of the circle circumference within the region on the total circumference : 

- Case A : Intersection with a single border

if $e_1 < d$ and $e_2 \geq d$ : 

$$w_{ij} = 1 - \frac{cos^{-1}(\frac{e_1}{d})}{\pi} \quad  \left( + \quad \frac{e_1\sqrt {d^2 - e_1^2}}{\pi d^2}\right)$$

- Case B : Intersection with two borders, excluding the corner

if $e_1 < d, e_2 < d$ and $e_1^2 + e_2^2 \geq d^2$ :

$$w_{ij} = 1 - \frac{cos^{-1}(\frac{e_1}{d}) + cos^{-1}(\frac{e_2}{d})}{\pi} \quad  \left( + \quad \frac{e_1\sqrt {d^2 - e_1^2} + e_2\sqrt{d^2 - e_2^2}}{\pi d^2}\right)$$

- Case C : Intersection with two borders, including the corner

if $e_1 < d, e_2 < d$ and $e_1^2 + e_2^2 < d^2$ :

$$w_{ij} = \frac{3}{4} - \frac{cos^{-1}(\frac{e_1}{d}) + cos^{-1}(\frac{e_2}{d})}{2\pi} \quad  \left( + \quad \frac{2e_1e_2 + e_1\sqrt {d^2 - e_1^2} + e_2\sqrt{d^2 - e_2^2}}{2\pi d^2}\right)$$

- Case D : No intersection with any border

$$w_{ij} = 1$$

The Ripley's Correction (RC) corresponds to the calculate part of the weight without parentheses. The Besag Correction (BC) corresponds to all the calculate part including terms inside and outside parentheses. The most common to use is RC but the BC has been also created to reduce the possible impact of the weight calculated by RC by taking the area information instead of the cirumference. (I'm not speaking about code but the reason of the creation of BC).

### Wiegand & Moloney correction function (WM's correction), a numerical approach

For our studied cases, we will probably need to compute a correction method which can calculate the weight of the Ripley's functions by approaching the analytical implementation by focus on pixel.

To keep it simple, The WM's correction, is the numerical implementation of the Besag's correction (BC) because we're counting the number of pixels inside the area on the number of pixel that constitute our circle.
To check if a pixel, which belongs to the circle, belongs either to the area.

To do it we first, generate our circle (of radius h) and center on the related event coordinates and store all the pixels's coordinates within the generated circle. After that, we can extract a sub area of the global binary matrix which represents the '.tif' file area where we're sure to find our generated circle.

Then we can compare the pixels's circle coordinates between the sub area extracted. To count the pixels of the circle inside the area we just have to check the binary value of the this pixel inside the sub area. If we have 1, this circle pixel belongs to the area, otherwise we have a 0 that means that isn't belongs to the area.

We finish it by doing the ratio of the number of pixels's circle inside the area on the total number of circle's pixels.

$$ w_{ij} = \frac{N(rgb(C_{i}) | R)}{N(rgb(C_{i}))} $$

with $N(rgb(C_{i}) | R)$ representing the number of pixels's circle inside the area and $N(rgb(C_{i}))$ representing the total number of circle's pixels.

In [8]:
# Function to calculate distance between all pairs points model
def calculate_distances(points):
    """
    Parameters:
        points : A 2D list of generated points (from the model)

    Returns:
        distances : Matrix which store the distance between each pair points
    """
    
    # Length of the list of points
    n = len(points)
    # Matrix to stored the distance values between point i and point j
    distances = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            # Calculation of the Euclidean distance between i and j
            distances[i, j] = np.linalg.norm(points[i] - points[j])
    # Return the matrix of distances
    return distances

# Indicator function according to distance h
def indicator_function(distances, h):
    """
    Parameters:
        distances : matrix of the distance of all existing pairs points
        h : distance of the radius of the K-function

    Returns:
        indicator_matrix : matrix filled by 0 and 1 and it depends on the following condition
    """
    
    # Return 1 if the condition is TRUE, 0 if it's FALSE
    return (distances <= h).astype(int)

# Function used to calculate the value e1 and e2 important for the Ripley's correction
def calculate_e1_e2(point, region):
    """
    Parameters:
        point : Coordinates of a model point
        region : list of for value of each side of the area (square or rectangle)

    Returns:
        e1 : minimum value between our point and one side of the region
        e2 : second minimum value between our point and another side of the region
    """
    
    # Recovery of the x-axis in x and y-axis in y which represent the coordinates of the point
    x, y = point
    
    # Limits coordinates of our region
    x_min, x_max, y_min, y_max = region
    
    # Sampling of the minimum distance between the point and a side
    e1 = min(x - x_min, x_max - x, y - y_min, y_max - y)
    # Distances between the point and each side of the region
    distances = [x - x_min, x_max - x, y - y_min, y_max - y]
    # REmoving the minimum value already stored
    distances.remove(e1)
    # Sampling of the minimum distance between the point and a side
    e2 = min(distances)
    
    # Return the two minimum distances between a side and the studied point
    return e1, e2

# Function to calculate the weight/influence of every point in relation to the edge effects problems (Ripley's Correction)
def calculate_weight_Ripley(point, region, h):
    """
    Parameters:
        point : Coordinates of a model point
        region : list of for value of each side of the area (square or rectangle)
        h : distance of the radius of the K-function
        
    Return:
        wij : Value of the Ripley's weight of the selected point"""

    # Calculation of the two minimum distances between a side of the region and the selected point
    e1, e2 = calculate_e1_e2(point, region)
    
    # Calculation of the Proportion of Circle Circumference of the point regarding the following cases
    # Intersection with a single border
    if (e1 < h and e2 >= h) or (e1 >= h and e2 < h):  # Case A
        w_ij = 1 - (np.arccos(e1/h) / np.pi) if e1 < h else 1 - (np.arccos(e2/h) / np.pi)
    elif e1 < h and e2 < h:
        # Intersection with two borders, excluding the corner
        if e1**2 + e2**2 >= h**2:  # Case B
            w_ij = 1 - (np.arccos(e1/h) + np.arccos(e2/h)) / np.pi
        # Intersection with two borders, including the corner
        else:  # Case C
            w_ij = 3/4 - (np.arccos(e1/h) + np.arccos(e2/h)) / (2 * np.pi)
    # No edge effects applied on the point
    else:   # Case D
        w_ij = 1.0  # No intersection with border
        
    # The weight of our point is the ratio between circle's circumference inside the area on the total circle's circumference
    return w_ij

# Function to calculate the weight/influence of every point in relation to the edge effects problems (Besag's Correction)
def calculate_weight_Besag(point, region, h):
    """
    Parameters:
        point : Coordinates of a model point
        region : list of for value of each side of the area (square or rectangle)
        h : distance of the radius of the K-function
        
    Return:
        wij : Value of the Besag's weight of the selected point"""

    # Calculation of the two minimum distances between a side of the region and the selected point
    e1, e2 = calculate_e1_e2(point, region)
    
    # Calculation of the Proportion of Circle Circumference of the point regarding the following cases
    # Intersection with a single border
    if (e1 < h and e2 >= h) or (e1 >= h and e2 < h):  # Case A
        if e1 < h:
            w_ij = 1 - (np.arccos(e1/h) / np.pi) + e1 * np.sqrt(h**2 - e1**2) / (np.pi * h**2)
        else:
            w_ij = 1 - (np.arccos(e2/h) / np.pi) + e2 * np.sqrt(h**2 - e2**2) / (np.pi * h**2)
    elif e1 < h and e2 < h:
        # Intersection with two borders, excluding the corner
        if e1**2 + e2**2 >= h**2:  # Case B
            w_ij = 1 - (np.arccos(e1/h) + np.arccos(e2/h)) / np.pi + (e1 * np.sqrt(h**2 - e1**2) + e2 * np.sqrt(h**2 - e2**2)) / (np.pi * h**2)
        # Intersection with two borders, including the corner
        else:  # Case C
            w_ij = 3/4 - (np.arccos(e1/h) + np.arccos(e2/h)) / (2 * np.pi) + (2 * e1 * e2 + e1 * np.sqrt(h**2 - e1**2) + e2 * np.sqrt(h**2 - e2**2)) / (2 * np.pi * h**2)
    # No edge effects applied on the point
    else:   # Case D
        w_ij = 1.0  # No intersection with border
        
    # The weight of our point is the ratio between circle's circumference inside the area on the total circle's circumference
    return w_ij

# Function to calculate the weight of each point using Wiegand and Moloney correction (Besag's numeric approach)
def calculate_weight_Wiegand_Moloney(point, img_array_area, h):
    """
    Parameters:
        point : coordinates of the point (event)
        img_array_area : Binary matrix of the area (1 represents the area, 0 represents outside)
        h : Distance of the radius of the K-function

    Returns:
        weight : weight for our point
    """
    # Get the whole dimensions of the image
    n_rows, n_cols = img_array_area.shape
    
    # Calculate the number of pixels in a circle with radius h
    # Coordinates of the regular area which contain the circle
    y, x = np.ogrid[-h:h+1, -h:h+1]
    # Generate a binary matrix to know the total number of pixels which represent the circle
    mask = x**2 + y**2 <= h**2
    # Sum all pixels of the subregion which has 1 in value
    num_pixels_in_circle = np.sum(mask)
    
    # Coordinates of our point
    x_center, y_center = point
    # Extract the sub-array around the point of side with minimum 2h (make sure that the following boundary values are int) 
    x_min = int(max(x_center - h, 0))
    x_max = min(x_center + h, n_cols)
    y_min = int(max(y_center - h, 0))
    y_max = min(y_center + h, n_rows)
    
    # Conditions to be sure than all the circle is taking into account for the weight ratio
    if x_max > int(np.round(x_max)):
        x_max = int(x_max) + 1
    else:
        x_max = int(x_max)
    if y_max > int(np.round(y_max)):
        y_max = int(y_max) + 1
    else:
        y_max = int(y_max)
    
    # Extract a sub area which contains the circle
    sub_array = img_array_area[y_min:y_max, x_min:x_max]
    
    # Be sure to have the radius value as an integer to pick up a correct value of bondary index
    if h > int(h):
        h = int(h)+1
    else:
        h = int(h)
    
    # Calculate the number of pixels in the circle within the area
    num_pixels_in_area_circle = np.sum(sub_array[mask[max(y_min-y_center+h,0):y_max-y_center+h, max(x_min-x_center+h,0):x_max-x_center+h]])
    # Calculate the weight
    weight = num_pixels_in_area_circle / num_pixels_in_circle
    
    # Return the weight value following the method
    return weight

# Function to calculate the weight of every model points with the application of the Ripley's Correction principe (take in count the edge effects)
def compute_weights(points, h, region, img_array_area, method):
    """
    Calculates the weights w_ij for each pair of points in the given region.

    Parameters:
    points : np.ndarray
        2D table of point coordinates.
    h : float
        Value of the K-function radius.
    region : list
        List [x_min, x_max, y_min, y_max] defining region boundaries.
    method : int
        Index of the correction method selected

    Returns:
    weights : np.value(n,n)
        Matrix of weights values of points i and j (wij <= 1).
    """
    
    # Initialisation of the matrix of weights with the basic case value (1)
    weights = np.ones((len(points),len(points)))
    
    # Determination of the weight w_ij which represents the ration of the circle circumference in the area on the total circle circumference
    # The center of the circle is point i with the associated radius d(i,j) 
    for i in range(len(points)):
        for j in range(len(points)):
            # Useless to calculate the weight of the circle with a radius d(i,i) = 0
            if i != j:
                # Coordinates of the point i
                s_i = points[i]
                # Coordinates of the point j
                s_j = points[j]
                # Weight of the point i related to the point j
                w_ij = compute_w_ij(s_i, s_j, h, region, img_array_area, method)
                # We're switching the initial weight's value with the real new one
                weights[i, j] = w_ij
    
    # Return the matrix of weights updated
    return weights

# Function to calculate the weight of the point s_i related to a point s_j
def compute_w_ij(s_i, s_j, h, region, img_array_area, method):
    """
    Calculates the weight w_ij for two points s_i and s_j,
    based on their distance and K-function's radius h.

    Parameters:
    s_i : list
        Coordinates of point s_i [x, y].
    s_j : list
        Coordinates of point s_j [x, y].
    h : float
        Radius h to define the region of interest (of K-function).
    region : list
        Defines the region boundaries [x_min, x_max, y_min, y_max].
    method : int
        Index of the correction method selected

    Returns:
    float
        Weight w_ij.
    """
    x_i, y_i = s_i
    x_j, y_j = s_j
    
    # Calculate the distance between points s_i and s_j
    distance_ij = math.sqrt((x_j - x_i)**2 + (y_j - y_i)**2)
    
    # If distance is greater than h, weight is 1.0 (Null)
    if distance_ij > h:
        return 1.0
    else:
        # Calculate the radius of the circle connecting s_i and s_j
        radius = distance_ij
        
        # Calculate the weight of the points pattern due to edge effects with a selected method
        if method == 0:
            # Calculate the part of the circle's circumference inside the region (Ripley's correction)
            w_ij = calculate_weight_Ripley(s_i,region,radius)
        elif method == 1:
            # Calculate the part of the circle's area inside the region (Besag's correction)
            w_ij = calculate_weight_Besag(s_i, region,radius)
        elif method == 2:
            w_ij = calculate_weight_Wiegand_Moloney(s_i, img_array_area, radius)
        else:
            # Wrong index of method, no method applied
            print("No Correction method selected")
            w_ij = 1.0
        
        # Return the weight of each point related to edge effects
        return w_ij

# Function to calculate the K-value with known fixed h with the edge effects corrections
def Correction_K_function(points, region, h, img_array_area, method, hull=None):
    """
    Parameters:
        points : A 2D list of generated points (from the model)
        region : list of for value of each side of the area (square or rectangle)
        h : distance of the radius of the K-function
        method : Index of the correction method selected
    
    Returns:
        K_corr : Value of the corrected K-function (with Ripley's correction)
        K_norm : Value of the uncorrected K-function (without Ripley's correction)
        weights : Matrix of weights of the point i (center) and j (arc)
        w : Matrix of the weight of point i and j without edge effects (d(i,j) = 1)"""
    
    # Length of the list of points
    n = len(points)
    
    # Calculation of the distances of our model points
    distances = calculate_distances(points)
    
    # Implementation of all weights related to Ripley's correction or another selected method (weights) and without it (w)
    w = np.ones((len(points), len(points)))  
    weights = compute_weights(points, h, region, img_array_area, method)
    
    # Create a mask for distances less than or equal to h and greater than 0 to exclude self-distances
    mask = (distances <= h) & (distances > 0)
    
    # Calculate K_corrected and K_normal based on the mask and weights
    K_corrected = np.sum(mask / weights)
    K_normal = np.sum(mask)
    
    # Possibility to calculate the area of the events for K-function values
    if hull is not None:
        # Calculate K values
        K_corrected = (hull.area / n**2) * K_corrected
        K_normal = (hull.area / n**2) * K_normal
    else:
        # Regular area value
        area_R = (region[1] - region[0]) * (region[3] - region[2])
        # Calculate K values
        K_corrected = (area_R / n**2) * K_corrected
        K_normal = (area_R / n**2) * K_normal
    
    # Return Corrected and Uncorrected values of K-function and the associated weight's list of each point
    return K_corrected, K_normal, weights, w

# Function to display the correction method selected impact and the weight of all model points
def display_kernel_weights(points, weights, region, radius):
    """
    Parameters:
        points : A 2D list of generated points (from the model)
        weights : List of the weight of each point
        region : list of for value of each side of the area (square or rectangle)
        radius : distance of the radius of the K-function
    """

    plt.figure(figsize=(10, 8))
    
    # Sum all the weights for each point
    weight_list = np.sum(weights, axis=1)
    
    # Plot the model
    scatter = plt.scatter(points[:, 0], points[:, 1], c=weight_list, cmap='coolwarm', edgecolor='black', label='Point')
    plt.colorbar(scatter, label='Weight')
    
    # Plot the limits of the region/area for a regular one
    plt.axhline(region[2] + radius, linestyle='--', c='yellow')
    plt.axhline(region[3] - radius, linestyle='--', c='yellow')
    plt.axvline(region[0] + radius, linestyle='--', c='yellow')
    plt.axvline(region[1] - radius, linestyle='--', c='yellow')

    # Set title of the graph
    plt.title('Weight value of points model in relation to K-function radius h')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.legend()
    plt.show()
    
# Function to display the correction method selected impact and the weight of all model points
def display_kernel_weights_bis(points, weights, outline):
    """
    Parameters:
        points : A 2D list of generated points (from the model)
        weights : List of the weight of each point
        outline : circumrference coordinates of the irregular area
    """

    plt.figure(figsize=(10, 8))
    
    # Sum all the weights for each point
    weight_list = np.sum(weights, axis=1)
    
    # Plot the model
    scatter = plt.scatter(points[:, 0], points[:, 1], c=weight_list, cmap='coolwarm', edgecolor='black', label='Point')
    plt.colorbar(scatter, label='Weight')

    # Plot the contour within the region of interest for irregular one
    adjusted_outline = [np.array([[y, x] for x, y in contour]) for contour in outline]
    for a_outline in adjusted_outline:
        plt.plot(a_outline[:, 0], a_outline[:, 1], linewidth=2, color='blue', label='Contours')

    # Set title of the graph
    plt.title('Weight value of points model in relation to K-function radius h')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.legend()
    plt.show()

### Ripley's functions (F,G,K,L,J...) under a correction method (CM)

Once these weights calculated and stored, we can calculate the corrected value of the K-function where we know the general formula and the corrected formula :

$$K(h) = \frac{1}{n\lambda} \sum_{i=1}^{n} \sum_{j \neq i} I_h (d_{ij} < h)$$

where $h$ is the radius K-function, $n$ the number of pattern points, $\lambda$ the Poisson parameter linked to the model, $I_h$ an indicator function with the distance $d_{ij}$.
For the corrected K-function value, we add the weight in the formula :

$$K_{CM}(h) = \frac{1}{n\lambda} \sum_{i=1}^{n} \sum_{j \neq i} \frac{I_h (d_{ij} < h)}{w_{ij}}$$

We can also do it for the L-function because is calculated according to the K-function :

$$L(h) = \sqrt \frac{K(h)}{\pi}$$

So, if we want the L-function value with a CM for the edge effects we just have to do :

$$L_{CM}(h) = \sqrt \frac{K_{CM}(h)}{\pi}$$

For the 3 others Ripley's function (F,G,J) it's under process...

In [9]:
# Calculation of the K-function values under a corrected condition
def weighted_ripley_k(points, max_dist, region, img_array_area, method, hull_pattern, support=40):
    """
    Calculate the weighted Ripley's K function for a set of points with weights.
    
    Parameters:
        points : A 2D numpy array of points.
        max_dist : The maximum distance to calculate K(d).
        region : list of for value of each side of the area (square or rectangle)
        img_array_area : Binary matrix to represent the data area
        method : Index of the selected correction method
        support : Number of distances to evaluate.
    
    Returns:
        distances : the evaluated distances h divide by support value, 
        K_values : the corresponding K(d) values with the correction method specify.
        K_norm : the corrsponding K(d) values without a correction method applied.
    """
    
    # Division of the distances between 0 and the max kNN value by support
    distances = np.linspace(0, max_dist, support)
    # Initialisation of the K_{RC}(d) values
    K_values = np.zeros(support)
    K_norm = np.zeros(support)
    
    # Calculation of the the K-function RC value following the previous formula :
    for i, h in enumerate(distances):
        # Update of the weight's matrix according the h K-function radius
        K_corr, K_wout, _, _ = Correction_K_function(points, region, h, img_array_area, method, hull_pattern)
        # Sub the initialized values by the one from the formula
        K_values[i] = K_corr
        K_norm[i] = K_wout
    # Return the distances where K-values has been calculated
    return distances, K_values, K_norm

# Calculation of the L-function values under a corrected condition
def weighted_ripley_l(K_values, K_norm):
    """
    Parameters:
        K_values : Values of the weighted K-function
        K_norm : Values of the normal K-function
        
    Return:
        L_values : Values of the weighted L-function
        L_norm : Values of the normal L-function
    """
    
    # L-function values related to the L formula
    L_values = np.sqrt(K_values / np.pi)
    L_norm = np.sqrt(K_norm / np.pi)
    
    return L_values, L_norm

# Function to display the K Ripley's function as the function 'display_Ripley_K' by adding the plot of the observed K-values under RC
def display_Ripley_K_L_weight(points, weights, unweight, max_dist, region, method, img_array_area, data_info, hull_pattern=None, save_path='Data_Pasteur/16h29_5h18_exp191_rpr/Plotting_Analysis'):
    """
    Parameters:
        points : A 2D numpy array of generated points (from the model)
        weights : A 2D numpy array of weights with w[i,j] the weight of the point i related to the point j
        unweight : A 2D numpy array of weights (not applied) with w[i,j] = 1 value
        max_dist : maximum distance for Ripley's K function
        region : list of for value of each side of the area (square or rectangle)
        method : Index of correction method selected
        img_array_area : Binary matrix to represent the data area
        data_info : additional information to specify the data (e.g., 'data1', 'data2', etc.)
        hull_pattern: bounding box, scipy.spatial.ConvexHull, shapely.geometry.Polygon the hull used to construct a random sample pattern 
        save_path : path where the figure should be saved
    """
    
    # Ensure the save directory exists
    full_save_path = os.path.join(save_path, data_info)
    os.makedirs(full_save_path, exist_ok=True)

    # Calculate the weighted and unweighted Ripley K function
    distances, K_values, K_norm = weighted_ripley_k(points, max_dist, region, img_array_area, method, hull_pattern)

    # Calculate the Ripley K function furnished by 'pointpats' library
    K_test = pointpats.k_test(points, support=40, hull=hull_pattern, keep_simulations=True, n_simulations=1000)
    
    # Get significant distances where the observed G function is greater than all simulations
    significant_distances = get_significant_distances(K_test.support, K_test.statistic, K_test.simulations)
    
    # Print significant distances
    print("Significant distances where observed K function is greater than all simulations:")
    print(significant_distances)

    # Parameters for the display
    fig, ax = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw=dict(width_ratios=(4 ,6, 4)))
    
    # Gather all the weight value for each event
    unweight_list = np.zeros(len(points))
    for i in range(len(points)):
        unweight_list[i] = np.sum(unweight[i,:])
    
    # Plot the pattern itself on the left frame
    scatter = ax[0].scatter(points[:, 0], points[:, 1], c=unweight_list, cmap='viridis')
    ax[0].set_title("Pattern")
    plt.colorbar(scatter, ax=ax[0], label='No Weights')

    # Clean up labels and axes there, too 
    ax[0].set_xticks([])
    ax[0].set_yticks([])
    ax[0].set_xticklabels([])
    ax[0].set_yticklabels([])
    
    
    # Plot all the simulations with very fine lines (black lines)
    ax[1].plot(K_test.support, K_test.simulations.T, color="k", alpha=0.01)
    
    # And show the average of simulations (blue line)
    ax[1].plot(
        K_test.support,
        np.median(K_test.simulations, axis=0),
        color="cyan",
        label="median simulation",
    )
    
    # And the observed pattern's K function (red line)
    ax[1].plot(K_test.support, K_test.statistic, label="observed", color="red")
    
    # Comparison with the analytic value (green line)
    ax[1].plot(distances, K_norm, label="observed (logic)", color="green")
    
    # Plot the observed pattern's weighted K function (orange line)
    ax[1].plot(distances, K_values, label="observed (weighted)", color="orange")
    
    # Clean up labels and axes
    ax[1].set_xlabel("Distance")
    ax[1].set_ylabel("K(d) value")
    ax[1].legend()
    ax[1].set_xlim(0, max_dist)
    
    # Select which correction is applied
    if method == 0:
        ax[1].set_title(r"Weighted Ripley's $K(d)$ function")
    elif method == 1:
        ax[1].set_title(r"Weighted Besag's $K(d)$ function")
    elif method == 2:
        ax[1].set_title(r"Weighted WM's $K(d)$ function")
    else:
        ax[1].set_title(r"Unweighted $K(d)$ function")
    
    # Gather all the weight value for each event
    weight_list = np.zeros(len(points))
    for i in range(len(points)):
        weight_list[i] = np.sum(weights[i,:])
    
    # Plot the pattern itself on the right frame
    scatter = ax[2].scatter(points[:, 0], points[:, 1], c=weight_list, cmap='viridis')
    ax[2].set_title("Pattern")
    plt.colorbar(scatter, ax=ax[2], label='Weights')

    # Clean up labels and axes there, too
    ax[2].set_xticks([])
    ax[2].set_yticks([])
    ax[2].set_xticklabels([])
    ax[2].set_yticklabels([])
    
    # Save the figure
    file_path = os.path.join(full_save_path, 'Weighted_K_function.png')
    plt.savefig(file_path)
    plt.show()
    
    # Calculate the L-function values from the K-function values
    L_values, L_norm = weighted_ripley_l(K_values, K_norm)
   
    # Calculation of the Ripley's L function
    l_test = pointpats.l_test(points, support=40, hull=hull_pattern, keep_simulations=True, n_simulations=1000)

    # Parameters for the display
    fig, ax = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw=dict(width_ratios=(4 ,6, 4)))
    
    # Gather all the weight value for each event
    unweight_list = np.zeros(len(points))
    for i in range(len(points)):
        unweight_list[i] = np.sum(unweight[i,:])
    
    # Plot the pattern itself on the left frame
    scatter = ax[0].scatter(points[:, 0], points[:, 1], c=unweight_list, cmap='viridis')
    ax[0].set_title("Pattern")
    plt.colorbar(scatter, ax=ax[0], label='No Weights')

    # Clean up labels and axes there, too
    ax[0].set_xticks([])
    ax[0].set_yticks([])
    ax[0].set_xticklabels([])
    ax[0].set_yticklabels([])

    # Plot all the simulations with very fine lines (black lines)
    ax[1].plot(l_test.support, l_test.simulations.T, color="k", alpha=0.01)
    
    # And show the average of simulations (blue line)
    ax[1].plot(
        l_test.support,
        np.median(l_test.simulations, axis=0),
        color="cyan",
        label="median simulation",
    )
    
    # And the observed pattern's L function (red line)
    ax[1].plot(l_test.support, l_test.statistic, label="observed", color="red")
    
    # Comparison with the analytic value
    ax[1].plot(distances, L_norm, label="observed (logic)", color="green")

    # Plot the observed pattern's weighted L function (orange line)
    ax[1].plot(distances, L_values, label="observed (weighted)", color="orange")

    # Clean up labels and axes
    ax[1].set_xlabel("Distance")
    ax[1].set_ylabel("L(d) value")
    ax[1].legend()
    ax[1].set_xlim(0, max_dist)
    
    # Select which correction is applied
    if method == 0:
        ax[1].set_title(r"Weighted Ripley's $L(d)$ function")
    elif method == 1:
        ax[1].set_title(r"Weighted Besag's $L(d)$ function")
    elif method == 2:
        ax[1].set_title(r"Weighted WM's $L(d)$ function")
    else:
        ax[1].set_title(r"Unweighted $L(d)$ function")

    # Gather all the weight value for each event
    weight_list = np.zeros(len(points))
    for i in range(len(points)):
        weight_list[i] = np.sum(weights[i,:])

    # Plot the pattern itself on the right frame
    scatter = ax[2].scatter(points[:, 0], points[:, 1], c=weight_list, cmap='viridis')
    ax[2].set_title("Pattern")
    plt.colorbar(scatter, ax=ax[2], label='Weights')

    # Clean up labels and axes there, too
    ax[2].set_xticks([])
    ax[2].set_yticks([])
    ax[2].set_xticklabels([])
    ax[2].set_yticklabels([])
    
    fig.tight_layout()
    
    # Save the figure
    file_path = os.path.join(full_save_path, 'Weighted_L_function.png')
    plt.savefig(file_path)
    plt.show()

##### Comments : Focus on the results generated by the pointpats.distance_statistic library and the analytic one calculated by our own :

By dint of running and manipulating the above-mentioned functions calculated by the library in question or ourselves, we noticed a repeated problem with the output values :

For exemple with the K-function which is one of the most used and analyzed function by us, we had distinguished that the value calculated by our analytic function 'Correction_K_function' which returns K_values with the application of a corrected method and without it and the value calculated by the library's function 'k_test.statistic' (generated by pointpats.k_test(coordinates,...,...,...)) are differents :

$$K_{CM} \neq K(h) \neq K_{test}$$

Refering to our code, it's normal that $K_{CM} \neq K(h)$ because the first is the corrected value of the second where we apply the weight on the first and where we don't on the second. So we have $K_{CM} \geq K(h)$.

But the fact that $K(h) \neq K_{test}$ seems to be false. Because for both functions, we use the same data/coordinates and apply the same formula to calculate the K-function values. But, when we look at the display of both in the same graph, instead of being merged together, we can see that they've the same behaviour and the same way to increase but values are differents. 

We figure out why this unexpected difference occurs. If we look more in detail the formula that defines the K-function, by remplacing $\lambda = \frac{n}{A}$ by how it's calculated we have a new formula to analyze :

$$K(h) = \frac{1}{n\lambda} \sum_{i=1}^{n} \sum_{j \neq i} I_h (d_{ij} < h) = \frac{A}{n^2} \sum_{i=1}^{n} \sum_{j \neq i} I_h (d_{ij} < h)$$

with $n$ representing the number of events in the model, $d_{ij}$ the distance between points $i$ and $j$, $h$ the current K-function radius value to calculate the K-function with this radius. 

But now we have a new variable that wasn't present in the general formula of $K(h)$. It's the area parameter $A$. In documentation, we have determine that $A$ should be the area of the zone where we're studying the point pattern behaviour. And for analytic corrections this zone A is a regular one but based on the minimum and maximum coordinates of the outline of our studied area called after $R$. This area $R$ is a irregular one the most of the time and for analytic calculation we put it into regular form which includes the real irregular area $R$.

But by analyzing deeply how and why there was difference in K-function values implementation (between our function and library function), we have figured out that the reason is related by the selection of the area $A$ which is different. For us the area $A$ selected is the area where we study our point pattern so the region $R$ which is mainly irregular called $A(R)$ (but we can generate a regular area based on $R$ and which includes it called $A_r(R)$).

After the mutiple tests on differents area possible to have all the pattern model within the area, we have found which area $A$ is selected by the library algorithm function. For it, $A$ is the regular area generated on the minimum and maximum coordinates of the events that represent our pattern (called $A_r(pp)$). So we have mainly :

$$ A_r(R) \ge A_r(pp) \geq A(R)$$

where is not impossible to have $A_r(pp) \leq A(R)$, but we are sure that $A_r(R) \geq A(R)$ and $A_r(R) \geq A_r(pp)$.

That explain the difference of value between these two ways to calculate this K-function values.

For our analyses, we need to implement the K and L functions on the good area selection. Unfortunatly, the library used to do it selected an implicite area that didn't correspond to the one wanted as precised before.

To avoid this problem we only have to switch the area value inside of the library computation by doing a ratio between our wanted area on the library selected area that don't correspond at our waits.

The ratio $\eta$ correspond at :

$$ \eta = \frac{A(R)}{A_r(pp)} \qquad or \qquad \eta = \frac{A_r(R)}{A_r(pp)} $$

With this ratio applied on the K-function (L-function), we will have the K-value which will correspond to our area (mainly irregular) and dots data and not on the regular area generated/construct according our coordinates points pattern :

$$ K_{test} = \frac{A_r(pp)}{n^2} \sum_{i=1}^{n} \sum_{j \neq i} I_h (d_{ij} < h) $$

$$ K(h) = \eta \frac{A_r(pp)}{n^2} \sum_{i=1}^{n} \sum_{j \neq i} I_h (d_{ij} < h) = \frac{A(R)}{A_r(pp)} \frac{A_r(pp)}{n^2} \sum_{i=1}^{n} \sum_{j \neq i} I_h (d_{ij} < h) = \frac{A(R)}{n^2} \sum_{i=1}^{n} \sum_{j \neq i} I_h (d_{ij} < h) $$

After that we will have $K(h) = K_{test}$, we do the same for the simulation under CSR Hypothesis. It's the same idea for the L-function.

In [10]:
# Function to display the K Ripley's function as the function 'display_Ripley_K' by adding the plot of the observed K-values under RC
def display_Ripley_K_L_weight_scaled(points, weights, unweight, max_dist, region, method, img_array_area,data_info, hull_pattern=None, save_path='Data_Pasteur/16h29_5h18_exp191_rpr/Plotting_Analysis'):
    """
    Parameters:
        points : A 2D numpy array of generated points (from the model)
        weights : A 2D numpy array of weights with w[i,j] the weight of the point i related to the point j
        unweight : A 2D numpy array of weights (not applied) with w[i,j] = 1 value
        max_dist : maximum distance for Ripley's K function
        region : list of for value of each side of the area (square or rectangle)
        method : Index of correction method selected
        img_array_area : Binary matrix to represent the data area
        data_info : additional information to specify the data (e.g., data_number, 'data2', etc.)
        hull_pattern: bounding box, scipy.spatial.ConvexHull, shapely.geometry.Polygon the hull used to construct a random sample pattern 
        save_path : path where the figure should be saved
    """
    
    # Ensure the save directory exists
    full_save_path = os.path.join(save_path, data_info)
    os.makedirs(full_save_path, exist_ok=True)
    
    # Implementation of the library area implicitly selected
    library_region = [min(points[:,0]), max(points[:,0]), min(points[:,1]), max(points[:,1])]

    # Calculate the weighted and unweighted Ripley K function
    distances, K_values, K_norm = weighted_ripley_k(points, max_dist, region, img_array_area, method, hull_pattern)

    # Calculate the Ripley K function with the library function called 'k_test'
    K_test = pointpats.k_test(points, support=40, hull=hull_pattern, keep_simulations=True, n_simulations=1000)

    # Parameters for the display
    fig, ax = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw=dict(width_ratios=(4 ,6, 4)))
    
    # Gather all the weight value for each event
    unweight_list = np.zeros(len(points))
    for i in range(len(points)):
        unweight_list[i] = np.sum(unweight[i,:])
    
    # Plot the pattern itself on the left frame
    scatter = ax[0].scatter(points[:, 0], points[:, 1], c=unweight_list, cmap='viridis')
    ax[0].set_title("Pattern")
    plt.colorbar(scatter, ax=ax[0], label='No Weights')

    # Clean up labels and axes there
    ax[0].set_xticks([])
    ax[0].set_yticks([])
    ax[0].set_xticklabels([])
    ax[0].set_yticklabels([])
    
    # Calculate the region implicitly selected by the library 'pointpats' for the area
    library_area = (library_region[1] - library_region[0]) * (library_region[3] - library_region[2])

    # Creation of a new variable to store the updated simulations (with the change of area)
    K_updated_simulations = K_test.simulations * (hull_pattern.area / library_area)

    # Work on the updated simulations for the plotting (black lines)
    ax[1].plot(K_test.support, K_updated_simulations.T, color="k", alpha=0.01)

    # and print the median of all simulation (blue line)
    ax[1].plot(
        K_test.support,
        np.median(K_updated_simulations, axis=0),
        color="cyan",
        label="median simulation",
    )

    # Calculate as done before with the simulations the K-values with our own area selection (by using the ratio)
    K_observed_statistic = K_test.statistic * (hull_pattern.area / library_area)
        # and the function of the library K-function observed with the given pattern (red line)
    ax[1].plot(K_test.support, K_observed_statistic, label="observed", color="red")
    
    # Comparison with analytic value (green line)
    ax[1].plot(distances, K_norm, label="observed (logic)", color="green")
    
    # Plotting the weighted K-function of the observed pattern (orange line)
    ax[1].plot(distances, K_values, label="observed (weighted)", color="orange")
    
    # Clean up labels and axes there
    ax[1].set_xlabel("Distance")
    ax[1].set_ylabel("K(d) value")
    ax[1].legend()
    ax[1].set_xlim(0, max_dist)
    
    # Select which correction is applied
    if method == 0:
        ax[1].set_title(r"Weighted Ripley's $K(d)$ function")
    elif method == 1:
        ax[1].set_title(r"Weighted Besag's $K(d)$ function")
    elif method == 2:
        ax[1].set_title(r"Weighted WM's $K(d)$ function")
    else:
        ax[1].set_title(r"Unweighted $K(d)$ function")
    
    # Gather all the weight value for each event
    weight_list = np.zeros(len(points))
    for i in range(len(points)):
        weight_list[i] = np.sum(weights[i,:])
    
    # Plot the pattern itself on the right frame
    scatter = ax[2].scatter(points[:, 0], points[:, 1], c=weight_list, cmap='viridis')
    ax[2].set_title("Pattern")
    plt.colorbar(scatter, ax=ax[2], label='Weights')

    # Clean up labels and axes there
    ax[2].set_xticks([])
    ax[2].set_yticks([])
    ax[2].set_xticklabels([])
    ax[2].set_yticklabels([])
    
    fig.tight_layout()
    
    # Save the figure
    file_path = os.path.join(full_save_path, 'Weighted_scaled_K_function.png')
    plt.savefig(file_path)
    plt.show()
    
    # Get significant distances where the observed K function is greater than all simulations
    significant_distances = get_significant_distances(K_test.support, K_observed_statistic, K_updated_simulations)
    
    # Print significant distances
    print("Significant distances where observed K function is greater than all simulations:")
    print(significant_distances)
    
    # Get significant corrected distances where the observed K function is greater than all simulations
    significant_corrected_distances = get_significant_distances(K_test.support, K_values, K_updated_simulations)
    
    # Print significant distances
    print("Significant distances where observed weighted K function is greater than all simulations:")
    print(significant_corrected_distances)

    # Calculate the L-function values with/without correction weight
    L_values, L_norm = weighted_ripley_l(K_values, K_norm)
   
    # Calculate the L-function Ripley
    l_test = pointpats.l_test(points, support=40, hull=hull_pattern, linearized=True, keep_simulations=True, n_simulations=1000)

    # Parameters of the diplay
    fig, ax = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw=dict(width_ratios=(4 ,6, 4)))
    
    # Gather all the weight value for each event
    unweight_list = np.zeros(len(points))
    for i in range(len(points)):
        unweight_list[i] = np.sum(unweight[i,:])
    
    # Plot the pattern itself on the left frame
    scatter = ax[0].scatter(points[:, 0], points[:, 1], c=unweight_list, cmap='viridis')
    ax[0].set_title("Pattern")
    plt.colorbar(scatter, ax=ax[0], label='No Weights')

    # Clean up labels and axes there
    ax[0].set_xticks([])
    ax[0].set_yticks([])
    ax[0].set_xticklabels([])
    ax[0].set_yticklabels([])

    # Recover and store the L-simulated value to apply after the area correction with the ratio
    l_updated_simulation_tmp = (l_test.simulations + distances)**2 * np.pi * (hull_pattern.area / library_area)
    # Recalculate the real L-values for simulations after the selection of our area
    l_updated_simulations = np.sqrt(l_updated_simulation_tmp / np.pi) - distances
    # Plot all the simulations with very fine lines (black lines)
    ax[1].plot(l_test.support, l_updated_simulations.T, color="k", alpha=0.01)
    
    # and the median of the simulations (blue line)
    ax[1].plot(
        l_test.support,
        np.median(l_updated_simulations, axis=0),
        color="cyan",
        label="median simulation",
    )
    
    # Then do the same done for the L-simulations with the observed L-values
    l_observed_tmp = (l_test.statistic + distances)**2 * np.pi * (hull_pattern.area / library_area)
    l_observed_statistic = np.sqrt(l_observed_tmp / np.pi) - distances
    # Plot our observed L-function related to the pattern (red line)
    ax[1].plot(l_test.support, l_observed_statistic, label="observed", color="red")
    
    # Comparison with the analytic value (green line)
    ax[1].plot(distances, L_norm - distances, label="observed (logic)", color="green")

    # Plot the weighted L-function of the observed pattern (orange line)
    ax[1].plot(distances, L_values - distances, label="observed (weighted)", color="orange")

    # Clean up labels and axes there
    ax[1].set_xlabel("Distance")
    ax[1].set_ylabel("L(d) value")
    ax[1].legend()
    ax[1].set_xlim(0, max_dist)
    
    # Select which correction is applied
    if method == 0:
        ax[1].set_title(r"Weighted Ripley's $L(d)$ function")
    elif method == 1:
        ax[1].set_title(r"Weighted Besag's $L(d)$ function")
    elif method == 2:
        ax[1].set_title(r"Weighted WM's $L(d)$ function")
    else:
        ax[1].set_title(r"Unweighted $L(d)$ function")

    # Gather all the weight value for each event
    weight_list = np.zeros(len(points))
    for i in range(len(points)):
        weight_list[i] = np.sum(weights[i,:])

    # Plot the pattern itself on the right frame
    scatter = ax[2].scatter(points[:, 0], points[:, 1], c=weight_list, cmap='viridis')
    ax[2].set_title("Pattern")
    plt.colorbar(scatter, ax=ax[2], label='Weights')

    # Clean up labels and axes there
    ax[2].set_xticks([])
    ax[2].set_yticks([])
    ax[2].set_xticklabels([])
    ax[2].set_yticklabels([])
    
    fig.tight_layout()
    
    # Save the figure
    file_path = os.path.join(full_save_path, 'Weighted_scaled_L_function.png')
    plt.savefig(file_path)
    plt.show()
    
    # Return the list of the significant distances where we can have a clustered pattern
    return significant_distances, significant_corrected_distances

### Display of the kernel related to the model

In [11]:
# Function to plot the kernel density behaviour related to our model
def plot_kde_with_weights(points, weights, region, method):
    # Store all points in coordinates variables
    x, y = points.T
    # Store the boundary value of the area
    x_min, x_max, y_min, y_max = region
    # Generate the grid for the kernel density generation
    xx, yy = np.mgrid[x_min:x_max:100j, y_min:y_max:100j]
    positions = np.vstack([xx.ravel(), yy.ravel()])
    
    # Apply the kernel density on our point pattern
    kde = gaussian_kde(points.T, weights=np.sum(weights, axis=1))
    f = np.reshape(kde(positions).T, xx.shape)
    
    # Gather all the weight on each point
    weight_list = np.zeros(len(points))
    for i in range(len(points)):
        weight_list[i] = np.sum(weights[i,:])
    
    # Plot the KDE  
    plt.figure(figsize=(9, 9))
    plt.contourf(xx, yy, f, cmap='viridis_r', alpha=0.55)
    plt.colorbar(label='Level of the KDE')
    plt.scatter(x, y, c=weight_list, edgecolor='black')
    
    # Select the method correction
    if len(np.unique(weight_list,return_counts=False)) == 1:
        plt.title('KDE without Ripley\'s Correction')
    elif method == 0:
        plt.title('KDE with Ripley\'s Correction Weights')
    elif method == 1:
        plt.title('KDE with Besag\'s Correction Weights')
    elif method == 2:
        plt.title('KDE with WM\'s Correction Weights')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.show()

In [12]:
def cluster_and_display_points(points, significant_distances, method, data_info, save_path='Data_Pasteur/16h29_5h18_exp191_rpr/Plotting_Analysis'):
    """
    Clusters points based on the median of significant distances and displays the clusters.
    
    Parameters:
        points : A 2D array of points to be clustered
        significant_distances : A list of significant distances
        data_info : additional information to specify the data (e.g., 'data1', 'data2', etc.)
        save_path : path where the figure should be saved
    """
    
    # Case where there is no significant distances given
    if not significant_distances:
        print("No significant distances found. Clustering will not be performed.")
        return
    
    # Ensure the save directory exists
    full_save_path = os.path.join(save_path, data_info)
    os.makedirs(full_save_path, exist_ok=True)

    # Calculate the max of significant distances
    if method == 0 and len( significant_distances) > 1:
        distance = max(significant_distances)
        print("Max of significant distances from G-function:", distance)
        
    elif method == 1 and len(significant_distances) > 1:
        distance = min(significant_distances)
        print("Min of significant distances from K-function:", distance)
    
    elif method == 2 and len(significant_distances) > 1:
        distance = min(significant_distances)
        print("Min of significant distances from weighted K-function:", distance)
        
    else:
        print("No significant distances found. Clustering will not be performed.")
        return
    
    # Perform DBSCAN clustering based on the max distance
    clustering = DBSCAN(eps=distance, min_samples=2).fit(points)
    labels = clustering.labels_
    
    # Print the labels of each point
    print("Labels of each point:", labels)
    
    # Plot the clusters
    unique_labels = set(labels)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    plt.figure(figsize=(8, 6))
    for k, col in zip(unique_labels, colors):
        class_member_mask = (labels == k)
        xy = points[class_member_mask]
        plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=col, markeredgecolor='k', markersize=6)
    
    plt.title("Clusters based on significant distances")
    plt.xlabel("X Coordinates")
    plt.ylabel("Y Coordinates")
    
    # Save the figure
    file_path = os.path.join(full_save_path, 'DBSCAN_cluster.png')
    plt.savefig(file_path)
    plt.show()

### Function to generate our outline data according our irregular area in Polygon type to use our irregular area data

In [13]:
# Function to recover the outline of our model point pattern used to calculate the K-function value
def Hull_computation(area_region, dots_coordinates, img_array_area=None):
    # Switch of coordinates for the irregular area
    fixed_coordinates = np.array([dots_coordinates[:, 1], dots_coordinates[:, 0]]).T
    
    if img_array_area is None:
        # Bondary box (regular or irregular) for the implementation of K-function
        hull = Polygon([(area_region[0],area_region[2]),(area_region[1],area_region[2]),(area_region[1],area_region[3]),(area_region[0],area_region[3])])
        
        # Information linked to the point pattern model
        pp_r = PointPattern(dots_coordinates,window=hull)
        print("\n")
        pp_r.summary()
        
        # Return the regular boundary
        return hull, dots_coordinates
    else:
        # Figure out the contours of our area data
        outline = measure.find_contours(img_array_area, level=0.5)
        # Rectify the contour within the region of interest
        adjusted_outline = [np.array([[y, x] for x, y in outlines]) for outlines in outline]

        # Convert the outline into a list of points representing it
        points_list = []
        for outline in adjusted_outline:
            points_list.extend(outline)

        # Convert the list into array type with numpy
        points_array = np.array(points_list)

        # Create the ConvexHull from our events to create our outline's polygon
        hull_pattern_c = ConvexHull(points_array)

        # Generate the polygon which represents the irregular area of our model 
        polygon = Polygon(hull_pattern_c.points)

        # Check if all the points pattern are within or not the area
        points_inside = [Point(p).within(polygon) for p in fixed_coordinates]
        
        # Print the information of the points location compare the outline (out or in)
        if not all(points_inside):
            print("\nSome points are out of the hull")
            # Optionnal: print or not the points outside the area
            outside_points = [p for p, inside in zip(fixed_coordinates, points_inside) if not inside]
            print("Events out of the hull:", outside_points)
        else:
            print("\nAll events are inside the hull")
            
        
        # Information linked to the point pattern model
        pp_irr = PointPattern(dots_coordinates,window=polygon)
        pp_irr.summary()
        
        # Return the irregular bondary
        return polygon, fixed_coordinates

### Functions to extract data from 'tif' file and translation of these data into more manipulative variables

In [14]:
# Function to extract til file and display of it, with complements analysis
def image_tif_extraction(path, data_info, save_path='Data_Pasteur/16h29_5h18_exp191_rpr/Plotting_Analysis'):
    """
    Parameters:
        path : path of the tiff file to extract/analyze
        data_info : additional information to specify the data (e.g., 'data1', 'data2', etc.)
        save_path : path where the figure should be saved

    Returns:
        img_array_path : binary matrix of the image
    """
    
    # Ensure the save directory exists
    full_save_path = os.path.join(save_path, data_info)
    os.makedirs(full_save_path, exist_ok=True)
    
    # Image read from the path 
    img_path = tiff.imread(path)
    
    # Display of the image
    plt.title("Image of the area/events to study extracted from tiff file")
    plt.imshow(img_path)
    plt.xlabel("X Coordinates")
    plt.ylabel("Y Coordinates")
    
    # Determine the file name based on the presence of 'area' or 'dot' in the path
    if 'area' in path:
        file_name = 'area_data_file.png'
    elif 'dot' in path:
        file_name = 'dots_data_file.png'
    else:
        file_name = 'default_data_file.png'
    
    # Save the figure
    file_path = os.path.join(full_save_path, file_name)
    plt.savefig(file_path)
    plt.show()
    
    # Conversion of the image into a binary matrix
    img_array_path = np.array(img_path)
    
    # Obtain the different values informations (what value and how many)
    unique_values, counts = np.unique(img_array_path, return_counts=True)
    print(f"Unique values in the array: {unique_values}")
    print(f"Counts of each unique value: {counts}")
    
    # Switching value to substitute 255 to 1 (for binary value)
    img_array_path[img_array_path >= 1] = 1
    
    # Complementary analysis information related to the image        
    unique_values, counts = np.unique(img_array_path, return_counts=True)
    print(f"Unique values in the array after binarization: {unique_values}")
    print(f"Counts of each unique value after binarization: {counts}")
    
    # Return the binary matrix of the image
    return img_array_path

# Function to extract the outline of the zone
def outline_identification(img_array_area):
    """
    Perimeters:
        img_array_area : binary matrix of the image area

    Returns:
        outlines_circumference : array of the binary matrix with the coordintes of the outlines
    """
    
    # Find outlines at a constant value of 0.5 to find the boundary between 0 and 1
    outlines_circumference = measure.find_contours(img_array_area, level=0.5)

    # Adjust outlines for correct orientation by flipping the y-axis for the plotting
    adjusted_outlines = [np.array([[y, x] for x, y in outline]) for outline in outlines_circumference]

    # Plot the original image with the correct origin
    plt.imshow(img_array_area, cmap='gray', origin='upper')

    # Plot the outlines of the area
    for outline in adjusted_outlines:
        plt.plot(outline[:, 0], outline[:, 1], linewidth=2, color='blue')

    plt.title("Contours of the Area")
    plt.xlabel("X Coordinates")
    plt.ylabel("Y Coordinates")
    plt.show()
    
    # Return the first outlines array that could be modified later
    return outlines_circumference

# Function to figure out the events coordinates of the pattern
def dots_identification(img_array_dots, binary_value):
    """
    Parameters:
        img_array_dots : binary matrix of the image dots
        binary_value : the binary value of the events coordinates events
    Returns:
        coordinates : list of the coordinates of the events of the image extracted"""
        
    # Obtain the coordinates where an event is present (value 1)
    coordinates = np.where(img_array_dots == binary_value)
    coordinates = np.array(list(zip(coordinates[0], coordinates[1])))
    
    # Adjust y-coordinates to match the typical orientation
    adjusted_coordinates = np.array([coordinates[:, 1], img_array_dots.shape[0] - coordinates[:, 0]]).T
    
    # Plotting the points pattern of the image
    plt.scatter(adjusted_coordinates[:,0],adjusted_coordinates[:,1],c='blue',s=10,label='Data Points')
    plt.title("Point Process Extracted location")
    plt.xlabel("X Coordinates")
    plt.ylabel("Y Coordinates")
    plt.legend()
    plt.show()
    
    # Return coordinates of all the events
    return coordinates

# Function to display the area and the events inside it
def display_phenomena(img_array_area, img_array_dots, outlines, dots_coordinates, data_info, save_path='Data_Pasteur/16h29_5h18_exp191_rpr/Plotting_Analysis'):
    """
    Parameters:
        img_array_area : binary matrix of the image area
        img_array_dots : binary matrix of the image dots
        outlines : array of the binary matrix with the coordintes of the outlines
        dots_coordinates : list of the coordinates of the events of the image extracted
        data_info : additional information to specify the data (e.g., 'data1', 'data2', etc.)
        save_path : path where the figure should be saved
    """
    
    # Ensure the save directory exists
    full_save_path = os.path.join(save_path, data_info)
    os.makedirs(full_save_path, exist_ok=True)
    
    # Plot the area with the correct origin
    plt.imshow(img_array_area, cmap='gray', origin='upper')

    # Adjust outlines for correct orientation by flipping the y-axis for the plotting
    adjusted_outlines = [np.array([[y, x] for x, y in outline]) for outline in outlines]

    # Plot the contours
    for outline in adjusted_outlines:
        plt.plot(outline[:, 0], outline[:, 1], linewidth=2, color='blue', label='Contours')
        
    # Adjust y-coordinates to match the typical orientation
    adjusted_coordinates = np.array([dots_coordinates[:, 1], dots_coordinates[:, 0]]).T

    # Plot the points/events
    plt.scatter(adjusted_coordinates[:, 0], adjusted_coordinates[:, 1], c='red', s=10, label='Data Points')

    # Add title and labels
    plt.title("Contours and Points/Events")
    plt.xlabel("X Coordinates")
    plt.ylabel("Y Coordinates")
    plt.legend()
    plt.show()
    
    # Create a new figure
    plt.figure()

    # Plot the contours
    # Adjust contours for correct orientation by flipping the y-axis
    adjust_outlines = [np.array([[y, img_array_area.shape[0] - x] for x, y in outline]) for outline in outlines]

    for outline in adjust_outlines:
        plt.plot(outline[:, 0], outline[:, 1], linewidth=2, color='blue', label='Contours')

    # Switch of the coordinates values to correspond to the modeling outlines
    dots_coordinates = np.array([dots_coordinates[:, 1],img_array_dots.shape[0] - dots_coordinates[:, 0]]).T
    # Plot the points/events
    plt.scatter(dots_coordinates[:, 0], dots_coordinates[:, 1], c='red', s=10, label='Data Points')

    # Add title and labels
    plt.title("Contours and Points/Events")
    plt.xlabel("X Coordinates")
    plt.ylabel("Y Coordinates")
    plt.legend()
    # Save the figure
    file_path = os.path.join(full_save_path, 'display_phenomena.png')
    plt.savefig(file_path)
    plt.show()

# Function to correct/rectify the zone of the model     
def region_phenomena_determination(outlines_circumference,dots_coordinates):
    """
    Parameters:
        outlines_circumference : Outlines of our area data
        dots_coordinates : Coordinates of our events

    Returns:
        adjusted_outline : information of the outline of the area corrected to correspond to region
        fixed_coordinates : coordinates modified to correspond to the region limits
        region : box region of the area modified
        adjusted_region : region unmodified
    """
    
    # Assuming there's only one contour, if there are multiple you might need to handle them separately
    outline = outlines_circumference[0]

    # Find the bounding box of the contour
    min_y, min_x = np.min(outline, axis=0).astype(int)
    max_y, max_x = np.max(outline, axis=0).astype(int)
    
    # Extract the region of interest from the original image
    region = [0,max_x - min_x,0,max_y - min_y]
    adjusted_region = [min_x, max_x, min_y, max_y]

    # Coordinates modified to be inside the region
    fixed_coordinates = np.array([dots_coordinates[:, 1], dots_coordinates[:, 0]]).T
    
    # Adjust filtered coordinates to the region of interest
    fixed_coordinates[:, 0] -= min_x
    fixed_coordinates[:, 1] -= min_y

    # Change the contour to correspond to the region of interest
    adjusted_outline = np.array([[y - min_x, x - min_y] for x, y in outline])
    
    # Return the outline of the area, the events coordinates in the correct orientation of the x and y axis and the regular area dimension associated
    return adjusted_outline, fixed_coordinates, region, adjusted_region

### Executive function

In [15]:
def analysis_function(path_area, path_events, data_number, binary_value=0):
    # Extraction of the area information and plotting them
    img_array_area = image_tif_extraction(path_area,data_number)
    area_boundary = outline_identification(img_array_area)
    
    # Extraction of the dots model information and plotting them
    img_array_dots = image_tif_extraction(path_events,data_number)
    dot_coordinates = dots_identification(img_array_dots,binary_value=binary_value)
    
    # Display the area and the events in the same graph
    display_phenomena(img_array_area,img_array_dots,area_boundary,dot_coordinates,data_number)
    
    # Extraction of the regular area of the outlines recover from the phenomena to study
    area_circumference, dots_coordinates, area_dim, adjusted_dim = region_phenomena_determination(area_boundary,dot_coordinates)
    
    # K-function radius h selected to check the well implementation of the corrected methods
    h = 0.2 * min(area_dim[1],area_dim[3])
    
    # Bondary box (regular or irregular) for the implementation of K-function
    hull, _ = Hull_computation(area_dim,dots_coordinates)
    hull_irr, fixed_coordinates = Hull_computation(area_dim,dot_coordinates,img_array_area)
    
    print("\nAnalysis of our data pattern\n")
    
    # Calculation of the analytic K-function value with associated weight related to the method selected
    K_corr_WM, K_norm_irr, w_wm, _ = Correction_K_function(fixed_coordinates, adjusted_dim, h, img_array_area, 2, hull_irr) 
    K_corr_BC, _, w_bc, _ = Correction_K_function(dots_coordinates, area_dim, h, img_array_area, 1, hull)
    K_corr_RC, K_norm, w_rc, w = Correction_K_function(dots_coordinates, area_dim, h, img_array_area, 0, hull)

    # Printing of the K-function values (corrected or not)
    print(f"K-function value with WM's correction : {K_corr_WM}")
    print(f"K-function value without correction method : {K_norm_irr}")
    print(f"K-function value with Besag's correction : {K_corr_BC}")
    print(f"K-function value with Ripley's correction : {K_corr_RC}")
    print(f"K-function value without correction method : {K_norm}")

    # Calculation of the Nearest Neighbor Distances (NN-Distances) and their corresponding points indices
    nn_distances, indices_nn_d, pairs_points = unique_nearest_neighbor_distances(dots_coordinates)
    
    # Number of distances stored by NN-Distances list ('nn_distances', 'indices_nn_d')
    # m = len(nn_distances)
    # print(f"\nNombre de NN-Distances : {m}")
    # print(f"Nombre d'indices de ces distances : {len(indices_nn_d)}")
    # print(f"Nombre de couples d'indices de ces distances : {len(pairs_points)}")
    # print(f"Les données NN-distances uniques : {nn_distances}")
    # print(f"Les indices des ces NN-distances uniques : {indices_nn_d}")
    # print(f"Couples d'indices des nn-d distances : {pairs_points}")
    
    # Display of the model with weight correction possibly applied
    # Display of the kNN information
    display_spatial_points_analysis(dots_coordinates,area_dim,len(dots_coordinates),indices_nn_d,nn_distances,pairs_points)
    
    # Dispaly of the weight repatition in the model
    display_kernel_weights_bis(fixed_coordinates,w_wm,area_boundary)
    # display_kernel_weights(dots_coordinates,w_rc,area_dim,h)
    # display_kernel_weights(dots_coordinates,w_bc,area_dim,h)
    display_kernel_weights(dots_coordinates,w,area_dim,h)
    
    # Display of the quadrat method
    quadrat_method(dots_coordinates)
    
    # Detemination of the max value of kNN used in the plotting of K-function
    max_nn_d = max(nn_distances)
    
    # Display of the K an L functions with/without weight impact
    # display_Ripley_K_L_weight(fixed_coordinates,w_wm,w,max_nn_d,adjusted_dim,2,img_array_area,data_number,hull_pattern=hull_irr)
    # display_Ripley_K_L_weight(dots_coordinates,w_rc,w,max_nn_d,area_dim,0,img_array_area,data_number,hull_pattern=hull)
    # display_Ripley_K_L_weight(dots_coordinates,w_bc,w,max_nn_d,area_dim,1,img_array_area,data_number,hull_pattern=hull)
    # Ripley_dispaly_all(dots_coordinates, max_nn_d,data_number,hull=hull)
    sup_dist = display_Ripley_G(dots_coordinates,max_nn_d,data_number,hull_pattern=hull)
    
    print("Scaled area")
    
    # Display of the K and L functions with/without weight impact with the good area selection imput
    # display_Ripley_K_L_weight_scaled(dots_coordinates,w_rc,w,max_nn_d,area_dim,0,img_array_area,data_number,hull_pattern=hull)
    # display_Ripley_K_L_weight_scaled(dots_coordinates,w_bc,w,max_nn_d,area_dim,1,img_array_area,data_number,hull_pattern=hull)
    # display_Ripley_K_L_weight_scaled(fixed_coordinates,w_wm,w,max_nn_d,adjusted_dim,2,img_array_area,data_number,hull_pattern=hull)
    check_dist, corrected_dist =  display_Ripley_K_L_weight_scaled(fixed_coordinates,w_wm,w,max_nn_d,adjusted_dim,2,img_array_area,data_number,hull_pattern=hull_irr)
    
    # Comparison of the Kernel Density with/without the application of a method of correction
    # plot_kde_with_weights(dots_coordinates, w_rc, area_dim,0)
    # plot_kde_with_weights(dots_coordinates, w_bc, area_dim,1)
    plot_kde_with_weights(fixed_coordinates,w_wm, adjusted_dim,2)
    plot_kde_with_weights(dots_coordinates, w, area_dim,0)
    
    if len(sup_dist) < 2 and len(check_dist) < 2 and len(corrected_dist) < 2:
        cluster_and_display_points(fixed_coordinates,check_dist,1,data_number)
        print("Case 5:")
        print("This point pattern isn't a clustered model")
        return 5
    elif len(check_dist) < 2 and len(corrected_dist) >= 2:
        cluster_and_display_points(fixed_coordinates,corrected_dist,2,data_number)
        print("Case 3:")
        print("This point pattern could be a clustered model related to the correction method in K function")
        return 3
    elif len(sup_dist) < 2:
        cluster_and_display_points(fixed_coordinates,check_dist,1,data_number)
        print("Case 2:")
        print("This point pattern might be a clustered model related to the K function analysis")
        return 2
    elif len(corrected_dist) < 2:
        cluster_and_display_points(fixed_coordinates,sup_dist,0,data_number)
        print("Case 4:")
        print("This point pattern could be a clustered model related to the G function analysis")
        return 4
    else:
        cluster_and_display_points(fixed_coordinates,sup_dist,0,data_number)
        print("Case 1:")
        print("This point pattern is a clustered model")
        return 1
 
# Function to extract the name/label of the data   
def extract_data_info(file_path):
    # Define the regex patterns to match 'expXXX' and 'fem_XX' separately
    exp_pattern = r'exp\d+'
    fem_pattern = r'fem_\d+'
    
    # Search for the first occurrence of each pattern in the file path
    exp_match = re.search(exp_pattern, file_path)
    fem_match = re.search(fem_pattern, file_path)
    
    if exp_match and fem_match:
        exp_part = exp_match.group(0)
        fem_part = fem_match.group(0)
        return f"{exp_part}_{fem_part}"
    else:
        return None

### Main function to execute the analysis of the pattern points of our study case (and also the simulation of random pattern inside the study area)

#### Data 1

In [16]:
# def main():
#     # Data 1
#     # Path of the file to extract and analyze
#     path_events = 'Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_3_z74-133_zprojMa_pouch_dots.tif'
#     path_area = 'Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_3_z74-133_zprojMax_areapouch.tif'
#     data_number = extract_data_info(path_area)
    
#     cluster_case = 0
#     random_case = 0
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 2
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_4_z74-128_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_4_z74-128_zprojMax_pouch_dots_good.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 3
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_5_z1-63_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_5_z1-63_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 4
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_6_z61-122_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_6_z61-122_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 5
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_7_z70-136_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_7_z70-136_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 6
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_8_z41-117_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_8_z41-117_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 7
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_9_z53-130_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_9_z53-130_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 8
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_10_z70-137_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_10_z70-137_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
    
#     # Data 9
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_11_z87-141_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_11_z87-141_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 10
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_12_z59-129_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_12_z59-129_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 11
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_13_56-128_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_13_56-128_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 12
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_14_z1-40_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_14_z1-40_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 13
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_15_z62-140_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-27_rpr_16h29_5h18_exp191_fem_15_z62-140_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 14
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_16_z1-45_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_16_z1-45_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 15
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_17_z1-56_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_17_z1-56_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 16
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_18_z49-107_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_18_z49-107_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 17
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_19_z1-43_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_19_z1-43_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 18
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_20_z1-42_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_20_z1-42_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 19
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_21_z44-113_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_21_z53-114_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 20
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_22_z43_109_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_22_z43_109_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 21
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_23_z53-114_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_23_z53-114_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 22
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_24_z57-111_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_24_z57-111_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 23
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_25_1-51_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_25_1-51_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 24
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_26_z53-98_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_26_z53-98_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 25
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_27_z52-108_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_27_z52-108_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 26
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_28_z46-103_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_28_z46-103_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     # Data 27
#     # Path of the file to extract and analyze
#     path_area = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_29_z48-101_zprojMax_areapouch.tif"
#     path_events = "Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/2024-05-28_rpr_16h29_5h18_exp191_fem_29_z48-101_zprojMax_pouch_dots.tif"
#     data_number = extract_data_info(path_area)
    
#     case_value = analysis_function(path_area, path_events, data_number, binary_value=0)
#     if case_value < 5:
#         cluster_case += 1
#     else:
#         random_case += 1
        
#     print(f"Pattern clustered detected : {cluster_case}")
#     print(f"Pattern random detected : {random_case}")
    
# if __name__ == "__main__":
#     main()

In [ ]:
def main():
    # Main path which contains experiments files
    directory_path = 'Data_Pasteur/16h29_5h18_exp191_rpr/16h29_5h18_exp191_rpr/'
    
    # Variable to store the maximum experiment index detected in the directory_path folder
    max_i = 0
    
    # Pattern to detect this max index
    pattern = re.compile(r'fem_(\d+)')
    
    # Browse the folder to find the maximum index value
    for filename in os.listdir(directory_path):
        match = pattern.search(filename)
        if match:
            fem_i_value = int(match.group(1))
            if fem_i_value > max_i:
                max_i = fem_i_value
    print(f"Max value fem : {max_i}")
    
    # Experiences files identified with fem_i
    experiences = [f"fem_{i}_" for i in range(2,max_i + 1)]
    
    # Count of clustered and random model
    cluster_case = 0
    random_case = 0
    
    # Run it for each experiment
    for experience in experiences:
        area_file = None
        events_file = None

        # Find the correct files which contain fem_i in the folder
        for filename in os.listdir(directory_path):
            # Verify if the file has the ID of the selected fem_i
            if experience in filename:
                # Verify if the file has to be given to the area data or events data
                if 'areapouch' in filename:
                    area_file = os.path.join(directory_path, filename)
                elif 'dots' in filename:
                    events_file = os.path.join(directory_path, filename)
        
        # Be sure that the generated files path match for area and dots (both exists)
        if area_file and events_file:
            # Name of the experiment folder's outputs
            data_number = extract_data_info(area_file)

            # Execution of the analysis
            case_value = analysis_function(area_file, events_file, data_number, binary_value=0)
            if case_value < 5:
                cluster_case += 1
            else:
                random_case += 1
        else:
            print(f"Missing files for the experiment : {experience}")

    print(f"Number of cluster_case: {cluster_case}")
    print(f"Number of random_case: {random_case}")
    
if __name__ == "__main__":
    main()

### Function on Ripley's functions K and L adaptation to match it with our data area

This following function will not be used for the data dur to its time execution (~30min by graph), but it's the more accurate code related to the area correction to calculate these Ripley's function (without the use of the 'pointpats' library which has defaults on the implicite area selection).

Here we only have done the area correction on the function $K$

In [18]:
# Function that tell us if our point to test is unique or not compare the other
def is_unique_point(new_point, points_list):
    """
    Parameters:
        new_point : Coordinates of a model point
        points_list : List of the points already stored and will compose the pattern model
    Returns:
        bool : True if the new_point is merged with a stored one, false otherwise
    """
    
    # Check if the point is merged with an other one
    return not any(np.array_equal(new_point, point) for point in points_list)

# Function to generate a random pattern inside the irregular area
def generate_random_points_within_contour(outline, num_points):
    """
    Parameters:
        outline : outline of the area
        num_points : number of points to generate

    Returns:
        Coordinates of points pattern inside the area (irregular or not)
    """
    
    # Min and max coordinates of the area contour
    min_y, min_x = np.min(outline, axis=0).astype(int)
    max_y, max_x = np.max(outline, axis=0).astype(int)
    
    # List to store the coordinates of the random pattern events
    points = []
    # Generate points randomly until we reach the number of points expected
    while len(points) < num_points:
        x = int(np.random.uniform(min_x, max_x))
        y = int(np.random.uniform(min_y, max_y))
        # Verifying that the point is in the area which are when we recover its coordinates has the (y for x and x for y)
        if measure.points_in_poly([[y, x]], outline) and is_unique_point([y,x], points):
            points.append([y, x])
    
    # Return the 2D list of coordinates of the events
    return np.array(points)

# Function to generate a random pattern inside a regular area
def generate_random_points_within_contour_regular(region, num_points):
    """
    Parameters:
        outline : outline of the area
        num_points : number of points to generate

    Returns:
        Coordinates of points pattern inside the area (irregular or not)
    """
    
    # Min and max coordinates of the area contour
    min_x, min_y = region[0], region[2]
    max_x, max_y = region[1], region[3]
    
    # List to store the coordinates of the random pattern events
    points = []
    # Generate points randomly until we reach the number of points expected
    while len(points) < num_points:
        x = int(np.random.uniform(min_x, max_x))
        y = int(np.random.uniform(min_y, max_y))
        # Check if the point is unique or not
        if is_unique_point([x,y],points):
            points.append([x, y])
    
    # Return the 2D list of coordinates of the events
    return np.array(points)

In [19]:
# Function to display the K Ripley's function as the function 'display_Ripley_K' by adding the plot of the observed K-values under RC
def display_Ripley_K_L_bis(points, weights, unweight, max_dist, region, method, img_array_area, outline, hull_pattern=None):
    """
    Parameters:
        points : A 2D numpy array of generated points (from the model)
        weights : A 2D numpy array of weights with w[i,j] the weight of the point i related to the point j
        unweight : A 2D numpy array of weights (not applied) with w[i,j] = 1 value
        max_dist : maximum distance for Ripley's K function
        region : list of for value of each side of the area (square or rectangle)
        method : Index of correction method selected
        img_array_area : Binary matrix which represent the data area
        outline : outline of our data area for the creation of a correct random pattern within this outline
        hull: bounding box, scipy.spatial.ConvexHull, shapely.geometry.Polygon the hull used to construct a random sample pattern 
    """

    # Calculate the weighted Ripley function K
    distances, K_values, K_norm = weighted_ripley_k(points, max_dist, region, img_array_area, method, hull_pattern)

    # Display parameters
    fig, ax = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw=dict(width_ratios=(4 ,6, 4)))
    
    # Weights summed for each point pattern
    unweight_list = np.zeros(len(points))
    for i in range(len(points)):
        unweight_list[i] = np.sum(unweight[i,:])
    
    # Plot the pattern itself on the left frame
    scatter = ax[0].scatter(points[:, 0], points[:, 1], c=unweight_list, cmap='viridis')
    ax[0].set_title("Pattern")
    plt.colorbar(scatter, ax=ax[0], label='No Weights')

    # Clean labels and axes
    ax[0].set_xticks([])
    ax[0].set_yticks([])
    ax[0].set_xticklabels([])
    ax[0].set_yticklabels([])
    
    # Set up the storage list of our own-generated simulations
    K_sim_all = np.zeros(40)
    # Set the number of simulation expexted (~2min for 10 simulations)
    for i in range(100):
        # Case where we're in a selected regular area
        if len(outline) == 4:
            random_pattern = generate_random_points_within_contour_regular(outline, len(points))
        # Case where we're in a selected irregular area
        else:
            random_pattern = generate_random_points_within_contour(outline, len(points))
        # Calculation of the K-function values needed to be plotted
        _, _, K_sim_norm = weighted_ripley_k(random_pattern,max_dist, region, img_array_area, method, hull_pattern)
        # Plot one by one all simulations with very light lines (black lines)
        ax[1].plot(distances, K_sim_norm, color="k", alpha=0.05)
        # Add each simulation value to the storage list
        K_sim_all += K_sim_norm
        # Print the number of simultion done (each print occurs after 10 simulations)
        if i%10 == 0:
            print(i)
    # When all simulations are done and plotted, we calculate the mean of all of them
    K_sim_all /= 100
    
    # And plot it (blue line)
    ax[1].plot(
        distances,
        K_sim_all,
        color="cyan",
        label="median simulation",
    )
    
    # Comparison with the analytic value (green line)
    ax[1].plot(distances, K_norm, label="observed (logic)", color="green")
    # Plot the weighted K-function of the observed pattern (orange line)
    ax[1].plot(distances, K_values, label="observed (weighted)", color="orange")
    
    # Clean labels and axes
    ax[1].set_xlabel("Distance")
    ax[1].set_ylabel("K(d) value")
    ax[1].legend()
    ax[1].set_xlim(0, max_dist)
    
    # Select title form the method selected
    if method == 0:
        ax[1].set_title(r"Weighted Ripley's $K(d)$ function")
    elif method == 1:
        ax[1].set_title(r"Weighted Besag's $K(d)$ function")
    elif method == 2:
        ax[1].set_title(r"Weighted WM's $K(d)$ function")
    else:
        ax[1].set_title(r"Unweighted $K(d)$ function")
    
    # Weights summed for each point pattern
    weight_list = np.zeros(len(points))
    for i in range(len(points)):
        weight_list[i] = np.sum(weights[i,:])
    
    # Plot the pattern itself on the right frame
    scatter = ax[2].scatter(points[:, 0], points[:, 1], c=weight_list, cmap='viridis')
    ax[2].set_title("Pattern")
    plt.colorbar(scatter, ax=ax[2], label='Weights')

    # Clean labels and axes
    ax[2].set_xticks([])
    ax[2].set_yticks([])
    ax[2].set_xticklabels([])
    ax[2].set_yticklabels([])
    
    # Show the K-function plotting
    fig.tight_layout()
    plt.show()

    # Calculate the L-function values from the K-function values
    L_values, L_norm = weighted_ripley_l(K_values, K_norm)
   
    # Calculation of the Ripley's L function
    l_test = pointpats.l_test(points, support=40, hull=hull_pattern, keep_simulations=True, n_simulations=1000)

    # Parameters for the display
    fig, ax = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw=dict(width_ratios=(4 ,6, 4)))
    
    # Gather all the weight value for each event
    unweight_list = np.zeros(len(points))
    for i in range(len(points)):
        unweight_list[i] = np.sum(unweight[i,:])
    
    # Plot the pattern itself on the left frame
    scatter = ax[0].scatter(points[:, 0], points[:, 1], c=unweight_list, cmap='viridis')
    ax[0].set_title("Pattern")
    plt.colorbar(scatter, ax=ax[0], label='No Weights')

    # Clean up labels and axes there, too
    ax[0].set_xticks([])
    ax[0].set_yticks([])
    ax[0].set_xticklabels([])
    ax[0].set_yticklabels([])

    # Plot all the simulations with very fine lines (black lines)
    ax[1].plot(l_test.support, l_test.simulations.T, color="k", alpha=0.01)
    
    # And show the average of simulations (blue line)
    ax[1].plot(
        l_test.support,
        np.median(l_test.simulations, axis=0),
        color="cyan",
        label="median simulation",
    )
    
    # And the observed pattern's L function (red line)
    ax[1].plot(l_test.support, l_test.statistic, label="observed", color="red")
    
    # Comparison with the analytic value
    ax[1].plot(distances, L_norm, label="observed (logic)", color="green")

    # Plot the observed pattern's weighted L function (orange line)
    ax[1].plot(distances, L_values, label="observed (weighted)", color="orange")

    # Clean up labels and axes
    ax[1].set_xlabel("Distance")
    ax[1].set_ylabel("L(d) value")
    ax[1].legend()
    ax[1].set_xlim(0, max_dist)
    
    # Select which correction is applied
    if method == 0:
        ax[1].set_title(r"Weighted Ripley's $L(d)$ function")
    elif method == 1:
        ax[1].set_title(r"Weighted Besag's $L(d)$ function")
    elif method == 2:
        ax[1].set_title(r"Weighted WM's $L(d)$ function")
    else:
        ax[1].set_title(r"Unweighted $L(d)$ function")

    # Gather all the weight value for each event
    weight_list = np.zeros(len(points))
    for i in range(len(points)):
        weight_list[i] = np.sum(weights[i,:])

    # Plot the pattern itself on the right frame
    scatter = ax[2].scatter(points[:, 0], points[:, 1], c=weight_list, cmap='viridis')
    ax[2].set_title("Pattern")
    plt.colorbar(scatter, ax=ax[2], label='Weights')

    # Clean up labels and axes there, too
    ax[2].set_xticks([])
    ax[2].set_yticks([])
    ax[2].set_xticklabels([])
    ax[2].set_yticklabels([])
    
    # Show the L-function plotting
    fig.tight_layout()
    plt.show()

### Cleaning all the generated files

If you want to erase some of your generated files, you need to execute the following code by writting in the main() the GOOD folder to erase and to input the GOOD command to execute the order to kill the content of this folder (command : 'KILL FOLDERS')

In [20]:
import shutil
import os

# Function to delete the generated files in the selected folders
def delete_folder_contents(folder_path):
    # Check if the folder exists and ends with 'Plotting_Analysis'
    if os.path.exists(folder_path) and os.path.isdir(folder_path) and folder_path.endswith('Plotting_Analysis'):
        # Delete all contents of the folder
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)  # Delete files and symbolic links
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)  # Delete directories and their contents
            except Exception as e:
                print(f'Error deleting {file_path}. Reason: {e}')
        print(f"Contents of the folder {folder_path} have been deleted.")
    else:
        print(f"The folder {folder_path} does not exist or does not end with 'Plotting_Analysis'.")

def main():
    # Prompt the user to enter the command
    command = input("Enter the command: ")
    
    if command == "KILL FOLDERS":
        folder_path = 'Data_pasteur/16h29_5h18_exp191_rpr/Plotting_Analysis'
        delete_folder_contents(folder_path)
    else:
        print("Command not recognized.")

if __name__ == "__main__":
    main()


Command not recognized.
